# Chapter 16: Storage

> *“Future users of large data banks must be protected from having to
> know how the data is organised in the machine (the internal
> representation).”*  
> — **Edgar F. Codd**, *A Relational Model of Data for Large Shared Data
> Banks* (1970)

------------------------------------------------------------------------

### 16.1 The Ephemeral Machine and the Persistent Domain

Throughout the preceding chapters, our computation has operated almost
exclusively within the transient theatre of volatile memory. When you
assign a new variable in Python or allocate a buffer on the C++ free
store via `std::make_unique`, you are leasing silicon real estate from
the operating system’s virtual memory subsystem. The nanosecond access
times of DRAM provide an exhilarating environment for raw computation,
but this speed comes at an absolute cost: **ephemerality**. Cut the
power rails, crash the operating system, or terminate the process host,
and the runtime state dissolves instantaneously into thermal entropy.

Software systems cannot function in a vacuum of transience. Banking
ledgers, social graphs, compiler symbol caches, and flight avionics
demand durability. When state transitions occur, they must endure
machine restarts, operating system panics, and catastrophic hardware
interrupts.

Historically, software engineers stored state by serialising in-memory
structures directly into flat files on disk. However, flat files
collapse the moment multiple processes read and write concurrently. Flat
files force the application layer to understand the byte-for-byte
physical layout of the hard drive: record offsets, padding bytes,
endianness, and disk block alignments. If the disk structure changes,
every line of application code breaks.

In 1970, an Oxford-educated mathematician working at IBM named **Edgar
F. Codd** published a landmark paper that severed this dependency
forever. Codd proposed the **Relational Model of Data**. He argued that
application programmers should never interact with pointers, disk
offsets, or physical memory layouts. Instead, data should be modelled
strictly as **mathematical relations** (sets of tuples), manipulated
through a formal **relational algebra**, and declared via high-level
non-procedural query languages. The underlying database engine was left
entirely free to lay out bytes on disk, balance index trees, and
optimise execution plans in whatever manner yielded the highest
throughput.

(See fig. 1 for an outline of this structure.)

<figure id="fig:storage-abstraction">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzNjhwdCIgaGVpZ2h0PSIxOTNwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzNjguMDAgMTkzLjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDE4
OSkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gYXBwbGljYXRpb24gLS0+CjxnIGlkPSJub2RlMSIg
Y2xhc3M9Im5vZGUiPgo8dGl0bGU+YXBwbGljYXRpb248L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJu
b25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjM2MCwtMTg1IDAsLTE4NSAwLC0xNDkgMzYwLC0x
NDkgMzYwLC0xODUiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTgwIiB5PSItMTcw
IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAu
MDAiPkFwcGxpY2F0aW9uIERvbWFpbjwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIg
eD0iMTgwIiB5PSItMTU5IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0i
IGZvbnQtc2l6ZT0iMTAuMDAiPihPYmplY3RzLCBTdHJ1Y3RzLCBBbGdlYnJhaWMgRGF0YSBUeXBl
cywgRHluYW1pYyBHcmFwaHMpPC90ZXh0Pgo8L2c+CjwhLS0gcmVsYXRpb25hbCAtLT4KPGcgaWQ9
Im5vZGUyIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5yZWxhdGlvbmFsPC90aXRsZT4KPHBvbHlnb24g
ZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIzNjAsLTEwNSAwLC0xMDUgMCwtNjkg
MzYwLC02OSAzNjAsLTEwNSIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxODAiIHk9
Ii05MCIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9
IjEwLjAwIj5SZWxhdGlvbmFsIEVuZ2luZSBDb3JlPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0i
bWlkZGxlIiB4PSIxODAiIHk9Ii03OSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8g
TWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj4oVHVwbGVzLCBSZWxhdGlvbnMsIFByZWRpY2F0ZXMs
IEFsZ2VicmFpYyBUcmFuc2Zvcm1zKTwvdGV4dD4KPC9nPgo8IS0tIGFwcGxpY2F0aW9uJiM0NTsm
Z3Q7cmVsYXRpb25hbCAtLT4KPGcgaWQ9ImVkZ2UxIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5hcHBs
aWNhdGlvbiYjNDU7Jmd0O3JlbGF0aW9uYWw8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJv
a2U9ImJsYWNrIiBkPSJNMTgwLC0xNDguNjlDMTgwLC0xMzguODkgMTgwLC0xMjYuNDIgMTgwLC0x
MTUuMzMiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTgz
LjUsLTExNS4yNiAxODAsLTEwNS4yNiAxNzYuNSwtMTE1LjI2IDE4My41LC0xMTUuMjYiLz4KPHRl
eHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSIxODAiIHk9Ii0xMzAiIGZvbnQtZmFtaWx5PSJJb3Nl
dmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+ICYjMTYwO09iamVjdCYj
NDU7UmVsYXRpb25hbCAvPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ic3RhcnQiIHg9IjE4MCIg
eT0iLTExOSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNp
emU9IjEwLjAwIj4gJiMxNjA7TWVtb3J5IEltcGVkYW5jZSBNaXNtYXRjaDwvdGV4dD4KPC9nPgo8
IS0tIHBoeXNpY2FsIC0tPgo8ZyBpZD0ibm9kZTMiIGNsYXNzPSJub2RlIj4KPHRpdGxlPnBoeXNp
Y2FsPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIz
NjAsLTM2IDAsLTM2IDAsMCAzNjAsMCAzNjAsLTM2Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRk
bGUiIHg9IjE4MCIgeT0iLTIxIiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRp
dW0iIGZvbnQtc2l6ZT0iMTAuMDAiPlBoeXNpY2FsIFN0b3JhZ2UgU3Vic3lzdGVtPC90ZXh0Pgo8
dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxODAiIHk9Ii0xMCIgZm9udC1mYW1pbHk9Iklv
c2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5CbG9jayBCdWZmZXJz
LCBXQUwsIEluZGljZXM6IEImIzQ1O1RyZWVzIC8gQlNUcywgRGlza3MpPC90ZXh0Pgo8L2c+Cjwh
LS0gcmVsYXRpb25hbCYjNDU7Jmd0O3BoeXNpY2FsIC0tPgo8ZyBpZD0iZWRnZTIiIGNsYXNzPSJl
ZGdlIj4KPHRpdGxlPnJlbGF0aW9uYWwmIzQ1OyZndDtwaHlzaWNhbDwvdGl0bGU+CjxwYXRoIGZp
bGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xODAsLTY4Ljc1QzE4MCwtNjIgMTgwLC01NC4w
OSAxODAsLTQ2LjU5Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2lu
dHM9IjE4My41LC00Ni40MyAxODAsLTM2LjQzIDE3Ni41LC00Ni40MyAxODMuNSwtNDYuNDMiLz4K
PHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSIxODAiIHk9Ii01MCIgZm9udC1mYW1pbHk9Iklv
c2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj4gJiMxNjA7UGFnZWQg
LyBSZWNvcmQgQWNjZXNzPC90ZXh0Pgo8L2c+CjwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 1: Levels of Storage Abstraction</figcaption>
</figure>

In this chapter, we bridge the gap between high-level language semantics
and low-level data persistence. We will explore:

1.  The mathematical underpinnings of **Relational Algebra**.
2.  **Indexing Theory**: transitioning from $O(N)$ full-scan sweeps to
    logarithmic $O(\log N)$ tree lookups.
3.  The **ACID** transaction model and the mechanics of atomic
    rollbacks.
4.  An end-to-end experimental implementation: constructing an in-memory
    relational database engine in **Modern C++20**, featuring a custom
    lexical analyser, recursive descent SQL parser, an indexing engine,
    and an algebraic execution pipeline.

### 16.2 Relational Algebra: The Mathematical Foundations

Before SQL was standardised by ANSI in 1986, relational databases were
grounded entirely in set theory and predicate logic. A database is not
fundamentally a collection of “tables” and “spreadsheets”; it is a set
of formal mathematical relations.

#### 16.2.1 Tuples, Relations, and Schemas

Let us define the relational model rigorously:

- **Domain ($D$):** A set of atomic values of a uniform data type (e.g.,
  $\mathbb{Z}$, $\mathbb{R}$, strings of ASCII characters, booleans).
- **Attribute ($A$):** A named label associated with a specific domain.
  For example, $\text{Age} \in \mathbb{N}$,
  $\text{Name} \in \text{String}$.
- **Relation Schema ($R$):** A finite list of attributes
  $R = (A_1: D_1, A_2: D_2, \dots, A_k: D_k)$. The integer $k$ is the
  **arity** (or degree) of the relation.
- **Tuple ($t$):** An ordered list of values $(v_1, v_2, \dots, v_k)$
  such that each $v_i \in D_i$. A tuple represents a single proposition
  or record.
- **Relation ($r(R)$):** A finite set of tuples sharing the schema $R$.
  In mathematical terms:
  $$r(R) \subseteq D_1 \times D_2 \times \dots \times D_k$$ Because a
  relation is fundamentally a **set**, two crucial mathematical
  properties follow:
  1.  **No Duplicates:** A relation cannot contain two identical tuples
      ($t_a = t_b \iff a = b$).
  2.  **Unordered:** The ordering of tuples within a relation has no
      semantic meaning.

#### 16.2.2 Fundamental Operators of Relational Algebra

Relational algebra is a **closed formal system**. Every relational
operator takes one or more relations as input and produces a brand-new
relation as output. This property of closure enables arbitrarily deep
nesting and composition of data queries (see fig. 2).

<figure id="fig:relational-algebra">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIyNDRwdCIgaGVpZ2h0PSIyMzlwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAyNDQuMDAgMjM5LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDIz
NSkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gciAtLT4KPGcgaWQ9Im5vZGUxIiBjbGFzcz0ibm9k
ZSI+Cjx0aXRsZT5yPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIg
cG9pbnRzPSIxNTIsLTIxNiA5OCwtMjE2IDk4LC0xOTQgMTUyLC0xOTQgMTUyLC0yMTYiLz4KPHBv
bHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxNTYsLTIyMCA5NCwtMjIw
IDk0LC0xOTAgMTU2LC0xOTAgMTU2LC0yMjAiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIg
eD0iMTI1IiB5PSItMjAyLjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1
bSIgZm9udC1zaXplPSIxMC4wMCI+QSB8IEIgfCBDPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0i
bWlkZGxlIiB4PSI3MSIgeT0iLTIyMyIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8g
TWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5SZWxhdGlvbiBSPC90ZXh0Pgo8L2c+CjwhLS0gam9p
biAtLT4KPGcgaWQ9Im5vZGUyIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5qb2luPC90aXRsZT4KPHBv
bHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIyMDgsLTE2OCAxMjIsLTE2
OCAxMjIsLTE0NiAyMDgsLTE0NiAyMDgsLTE2OCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxl
IiB4PSIxNjUiIHk9Ii0xNTQuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVk
aXVtIiBmb250LXNpemU9IjEwLjAwIj5OYXR1cmFsIEpvaW4gKOKLiCk8L3RleHQ+CjwvZz4KPCEt
LSByJiM0NTsmZ3Q7am9pbiAtLT4KPGcgaWQ9ImVkZ2UxIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5y
JiM0NTsmZ3Q7am9pbjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9
Ik0xMzcuMjIsLTE4OS45NUMxNDEuMDcsLTE4NS41MiAxNDUuMzcsLTE4MC41NyAxNDkuNDIsLTE3
NS45MiIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxNTIu
MzYsLTE3Ny44NyAxNTYuMjgsLTE2OC4wMyAxNDcuMDgsLTE3My4yOCAxNTIuMzYsLTE3Ny44NyIv
Pgo8L2c+CjwhLS0gc2VsZWN0aW9uIC0tPgo8ZyBpZD0ibm9kZTQiIGNsYXNzPSJub2RlIj4KPHRp
dGxlPnNlbGVjdGlvbjwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2si
IHBvaW50cz0iMjA4LC0xMjQgMTIyLC0xMjQgMTIyLC0xMDIgMjA4LC0xMDIgMjA4LC0xMjQiLz4K
PHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTY1IiB5PSItMTEwLjUiIGZvbnQtZmFtaWx5
PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+U2VsZWN0aW9u
ICjPgyk8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjY0IiB5PSItMTI3IiBm
b250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAi
PihGaWx0ZXIgUm93cyBieSBQcmVkaWNhdGUpPC90ZXh0Pgo8L2c+CjwhLS0gam9pbiYjNDU7Jmd0
O3NlbGVjdGlvbiAtLT4KPGcgaWQ9ImVkZ2UzIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5qb2luJiM0
NTsmZ3Q7c2VsZWN0aW9uPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIg
ZD0iTTE2NSwtMTQ1LjgzQzE2NSwtMTQyLjMyIDE2NSwtMTM4LjI2IDE2NSwtMTM0LjI0Ii8+Cjxw
b2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjE2OC41LC0xMzQuMDcg
MTY1LC0xMjQuMDcgMTYxLjUsLTEzNC4wNyAxNjguNSwtMTM0LjA3Ii8+CjwvZz4KPCEtLSBzIC0t
Pgo8ZyBpZD0ibm9kZTMiIGNsYXNzPSJub2RlIj4KPHRpdGxlPnM8L3RpdGxlPgo8cG9seWdvbiBm
aWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjIzMiwtMjE2IDE3OCwtMjE2IDE3OCwt
MTk0IDIzMiwtMTk0IDIzMiwtMjE2Ii8+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxh
Y2siIHBvaW50cz0iMjM2LC0yMjAgMTc0LC0yMjAgMTc0LC0xOTAgMjM2LC0xOTAgMjM2LC0yMjAi
Lz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjA1IiB5PSItMjAyLjUiIGZvbnQtZmFt
aWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+QyB8IEQg
fCBFPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxNTEiIHk9Ii0yMjMiIGZv
bnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+
UmVsYXRpb24gUzwvdGV4dD4KPC9nPgo8IS0tIHMmIzQ1OyZndDtqb2luIC0tPgo8ZyBpZD0iZWRn
ZTIiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPnMmIzQ1OyZndDtqb2luPC90aXRsZT4KPHBhdGggZmls
bD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTE5Mi43OCwtMTg5Ljk1QzE4OC45MywtMTg1LjUy
IDE4NC42MywtMTgwLjU3IDE4MC41OCwtMTc1LjkyIi8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBz
dHJva2U9ImJsYWNrIiBwb2ludHM9IjE4Mi45MiwtMTczLjI4IDE3My43MiwtMTY4LjAzIDE3Ny42
NCwtMTc3Ljg3IDE4Mi45MiwtMTczLjI4Ii8+CjwvZz4KPCEtLSBwcm9qZWN0aW9uIC0tPgo8ZyBp
ZD0ibm9kZTUiIGNsYXNzPSJub2RlIj4KPHRpdGxlPnByb2plY3Rpb248L3RpdGxlPgo8cG9seWdv
biBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjIwOCwtODAgMTIyLC04MCAxMjIs
LTU4IDIwOCwtNTggMjA4LC04MCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxNjUi
IHk9Ii02Ni41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQt
c2l6ZT0iMTAuMDAiPlByb2plY3Rpb24gKM+AKTwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9Im1p
ZGRsZSIgeD0iNjEiIHk9Ii04MyIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVk
aXVtIiBmb250LXNpemU9IjEwLjAwIj4oU2VsZWN0IFN1YnNldCBvZiBDb2x1bW5zKTwvdGV4dD4K
PC9nPgo8IS0tIHNlbGVjdGlvbiYjNDU7Jmd0O3Byb2plY3Rpb24gLS0+CjxnIGlkPSJlZGdlNCIg
Y2xhc3M9ImVkZ2UiPgo8dGl0bGU+c2VsZWN0aW9uJiM0NTsmZ3Q7cHJvamVjdGlvbjwvdGl0bGU+
CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xNjUsLTEwMS44M0MxNjUsLTk4
LjMyIDE2NSwtOTQuMjYgMTY1LC05MC4yNCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tl
PSJibGFjayIgcG9pbnRzPSIxNjguNSwtOTAuMDcgMTY1LC04MC4wNyAxNjEuNSwtOTAuMDcgMTY4
LjUsLTkwLjA3Ii8+CjwvZz4KPCEtLSBuZXcgLS0+CjxnIGlkPSJub2RlNiIgY2xhc3M9Im5vZGUi
Pgo8dGl0bGU+bmV3PC90aXRsZT4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTY1IiB5
PSItMTUuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNp
emU9IjEwLjAwIj5OZXcgUmVsYXRpb248L3RleHQ+CjwvZz4KPCEtLSBwcm9qZWN0aW9uJiM0NTsm
Z3Q7bmV3IC0tPgo8ZyBpZD0iZWRnZTUiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPnByb2plY3Rpb24m
IzQ1OyZndDtuZXc8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJN
MTY1LC01Ny42MUMxNjUsLTU0LjMzIDE2NSwtNTAuNTMgMTY1LC00Ni42MyIvPgo8cG9seWdvbiBm
aWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxNjguNSwtNDYuMzIgMTY1LC0zNi4z
MiAxNjEuNSwtNDYuMzIgMTY4LjUsLTQ2LjMyIi8+CjwvZz4KPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 2: Relational Operations as Data Queries</figcaption>
</figure>

There are six primitive operators defined by Codd:

**1. Selection ($\sigma$):**

The selection operator filters tuples from a relation $R$ that satisfy a
Boolean propositional formula $P$.

$$\sigma_P(R) = \{ t \in R \mid P(t) \text{ is true} \}$$

*Example:* Given an `Engineers` relation,
$\sigma_{\text{Age} \ge 21}(\text{Engineers})$ extracts all tuples where
the attribute `Age` is at least 21. In SQL, this corresponds directly to
the `WHERE` clause.

**2. Projection ($\pi$):**

The projection operator selects a subset of attributes from the relation
and discards the rest. If duplicate tuples emerge due to discarded
attributes, they are collapsed.

$$\pi_{A_1, A_2, \dots, A_m}(R) = \{ (t[A_1], t[A_2], \dots, t[A_m]) \mid t \in R \}$$

*Example:* $\pi_{\text{Name}, \text{Salary}}(\text{Engineers})$ outputs
a relation with an arity of 2, stripping away addresses, IDs, and ages.
In SQL, this corresponds to `SELECT Name, Salary`.

**3. Cartesian Product ($\times$):**

The Cartesian product (or cross product) combines every tuple of
relation $R$ with every tuple of relation $S$.

$$R \times S = \{ t \circ s \mid t \in R \land s \in S \}$$

If $|R| = N$ and $|S| = M$, then $|R \times S| = N \times M$. The schema
arity becomes $\text{arity}(R) + \text{arity}(S)$.

**4. Set Union ($\cup$):**

Combines all tuples from two relations. Both relations must be
**union-compatible** (they must have identical arity and matching
attribute domains).

$$R \cup S = \{ t \mid t \in R \lor t \in S \}$$

**5. Set Difference ($-$):**

Finds all tuples present in relation $R$ that do not exist in relation
$S$. Both relations must be union-compatible.

$$R - S = \{ t \mid t \in R \land t \notin S \}$$

**6. Rename ($\rho$):**

Renames attributes or the relation itself, avoiding naming collisions
during self-joins.

$$\rho_{S(B_1, \dots, B_k)}(R)$$

#### 16.2.3 Derived Operators: The Natural Join ($\bowtie$)

From these six primitive operators, we derive the most computationally
critical operator in database theory: the **Natural Join**.

A natural join ($R \bowtie S$) computes the Cartesian product
$R \times S$, selects tuples where all shared attributes match, and
projects away the duplicate common columns:

$$R \bowtie S = \pi_{\text{Attributes}}\left(\sigma_{R.A_1 = S.A_1 \land \dots \land R.A_m = S.A_m}(R \times S)\right)$$

### Relational Algebra to SQL Equivalence

The SQL language is simply a syntactic skin over relational algebra
expressions (see tbl. 1).

| Relational Algebra | SQL Construct | Time Complexity (Naive) |
|------------------------|------------------------|------------------------|
| $\sigma_{\text{Predicate}}(R)$ | `SELECT * FROM R WHERE Predicate` | $O(N)$ scan |
| $\pi_{\text{Cols}}(R)$ | `SELECT Cols FROM R` | $O(N)$ scan |
| $R \times S$ | `SELECT * FROM R CROSS JOIN S` | $O(N \times M)$ |
| $R \bowtie S$ | `SELECT * FROM R NATURAL JOIN S` | $O(N \times M)$ naive / $O(N \log M)$ indexed |
| $R \cup S$ | `SELECT * FROM R UNION SELECT * FROM S` | $O((N+M) \log(N+M))$ |

Table 1: Relation Algebra Expressions as SQL {#tbl:sq-language}

#### 16.3 The Anatomy of a Relational Database Engine

To translate declarative SQL text into concrete memory transformations,
a database engine operates as a multi-stage pipeline (see fig. 3).

<figure id="fig:db-engine">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIyMCA0MCA4ODAgMTMwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxMzAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiwgbW9ub3Nw
YWNlOyB9CiAgICA8L3N0eWxlPgogICAgPGxpbmVhckdyYWRpZW50IGlkPSJib3hHcmFkIiB4MT0i
MCUiIHkxPSIwJSIgeDI9IjEwMCUiIHkyPSIxMDAlIj4KICAgICAgPHN0b3Agb2Zmc2V0PSIwJSIg
c3RvcC1jb2xvcj0iIzFlMjkzYiIgLz4KICAgICAgPHN0b3Agb2Zmc2V0PSIxMDAlIiBzdG9wLWNv
bG9yPSIjMGYxNzJhIiAvPgogICAgPC9saW5lYXJHcmFkaWVudD4KICAgIDxsaW5lYXJHcmFkaWVu
dCBpZD0iYWNjZW50R3JhZCIgeDE9IjAlIiB5MT0iMCUiIHgyPSIxMDAlIiB5Mj0iMTAwJSI+CiAg
ICAgIDxzdG9wIG9mZnNldD0iMCUiIHN0b3AtY29sb3I9IiMzOGJkZjgiIC8+CiAgICAgIDxzdG9w
IG9mZnNldD0iMTAwJSIgc3RvcC1jb2xvcj0iIzAyODRjNyIgLz4KICAgIDwvbGluZWFyR3JhZGll
bnQ+CiAgICA8ZmlsdGVyIGlkPSJkcm9wU2hhZG93IiB4PSItMTAlIiB5PSItMTAlIiB3aWR0aD0i
MTIwJSIgaGVpZ2h0PSIxMzAlIj4KICAgICAgPGZlRHJvcFNoYWRvdyBkeD0iMiIgZHk9IjQiIHN0
ZERldmlhdGlvbj0iMyIgZmxvb2QtY29sb3I9IiMwMDAwMDAiIGZsb29kLW9wYWNpdHk9IjAuNCIv
PgogICAgPC9maWx0ZXI+CiAgPC9kZWZzPgogIDwhLS0gU3RhZ2UgMTogTGV4ZXIgJiBQYXJzZXIg
LS0+CiAgPGcgZmlsdGVyPSJ1cmwoI2Ryb3BTaGFkb3cpIj4KICAgIDxyZWN0IHg9IjMwIiB5PSI1
MCIgd2lkdGg9IjE1MCIgaGVpZ2h0PSIxMTAiIHJ4PSI4IiBmaWxsPSJ2YXIoLS1zdmctdGludC0x
LCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ry
b2tlLXdpZHRoPSIyIi8+CiAgICA8dGV4dCB4PSIxMDUiIHk9Ijg1IiBmaWxsPSJ2YXIoLS1zdmct
ZmcpIiBmb250LXNpemU9IjE2IiBmb250LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRs
ZSI+U1FMIFN0cmluZzwvdGV4dD4KICAgIDx0ZXh0IHg9IjEwNSIgeT0iMTE1IiBmaWxsPSJ2YXIo
LS1zdmctZmcpIiBmb250LXNpemU9IjE0IiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj5MZXhlciArIFBh
cnNlcjwvdGV4dD4KICAgIDx0ZXh0IHg9IjEwNSIgeT0iMTM1IiBmaWxsPSJ2YXIoLS1zdmctZmcp
IiBmb250LXNpemU9IjE0IiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj5Qcm9kdWNlcyBBU1Q8L3RleHQ+
CiAgPC9nPgogIDwhLS0gQXJyb3cgMSAtLT4KICA8cGF0aCBkPSJNIDE4MCAxMDUgTCAyMzAgMTA1
IiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMyIgZmlsbD0ibm9uZSIgbWFy
a2VyLWVuZD0idXJsKCNhcnJvd2hlYWQpIi8+CiAgPHBvbHlnb24gcG9pbnRzPSIyNDAsMTA1IDIz
MCwxMDAgMjMwLDExMCIgZmlsbD0idmFyKC0tc3ZnLWZnIi8+CiAgPCEtLSBTdGFnZSAyOiBMb2dp
Y2FsIFF1ZXJ5IFBsYW5uZXIgLS0+CiAgPGcgZmlsdGVyPSJ1cmwoI2Ryb3BTaGFkb3cpIj4KICAg
IDxyZWN0IHg9IjI0MCIgeT0iNTAiIHdpZHRoPSIxNjAiIGhlaWdodD0iMTEwIiByeD0iOCIgZmls
bD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9
InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIvPgogICAgPHRleHQgeD0iMzIwIiB5PSI4
NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNiIgZm9udC13ZWlnaHQ9ImJvbGQi
IHRleHQtYW5jaG9yPSJtaWRkbGUiPlF1ZXJ5IFBsYW5uZXI8L3RleHQ+CiAgICA8dGV4dCB4PSIz
MjAiIHk9IjExNSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgdGV4dC1hbmNo
b3I9Im1pZGRsZSI+TG9naWNhbCBQbGFuPC90ZXh0PgogICAgPHRleHQgeD0iMzIwIiB5PSIxMzUi
IGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIHRleHQtYW5jaG9yPSJtaWRkbGUi
PlJlbGF0aW9uYWwgQWxnIFRyZWU8L3RleHQ+CiAgPC9nPgogIDwhLS0gQXJyb3cgMiAtLT4KICA8
cGF0aCBkPSJNIDQwMCAxMDUgTCA0NTAgMTA1IiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9r
ZS13aWR0aD0iMyIgZmlsbD0ibm9uZSIvPgogIDxwb2x5Z29uIHBvaW50cz0iNDYwLDEwNSA0NTAs
MTAwIDQ1MCwxMTAiIGZpbGw9InZhcigtLXN2Zy1mZykiLz4KICA8IS0tIFN0YWdlIDM6IEV4ZWN1
dGlvbiBFbmdpbmUgLS0+CiAgPGcgZmlsdGVyPSJ1cmwoI2Ryb3BTaGFkb3cpIj4KICAgIDxyZWN0
IHg9IjQ2MCIgeT0iNTAiIHdpZHRoPSIxODAiIGhlaWdodD0iMTEwIiByeD0iOCIgZmlsbD0idmFy
KC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigt
LXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIvPgogICAgPHRleHQgeD0iNTUwIiB5PSI4NSIgZmls
bD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNiIgZm9udC13ZWlnaHQ9ImJvbGQiIHRleHQt
YW5jaG9yPSJtaWRkbGUiPkV4ZWN1dGlvbiBFbmdpbmU8L3RleHQ+CiAgICA8dGV4dCB4PSI1NTAi
IHk9IjExNSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgdGV4dC1hbmNob3I9
Im1pZGRsZSI+SXRlcmF0b3JzIC8gUGlwZWxpbmluZzwvdGV4dD4KICAgIDx0ZXh0IHg9IjU1MCIg
eT0iMTM1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiB0ZXh0LWFuY2hvcj0i
bWlkZGxlIj5TY2FuIC8gRmlsdGVyIC8gUHJvamVjdDwvdGV4dD4KICA8L2c+CiAgPCEtLSBBcnJv
dyAzIC0tPgogIDxwYXRoIGQ9Ik0gNjQwIDEwNSBMIDY5MCAxMDUiIHN0cm9rZT0idmFyKC0tc3Zn
LWZnKSIgc3Ryb2tlLXdpZHRoPSIzIiBmaWxsPSJub25lIi8+CiAgPHBvbHlnb24gcG9pbnRzPSI3
MDAsMTA1IDY5MCwxMDAgNjkwLDExMCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIvPgogIDwhLS0gU3Rh
Z2UgNDogU3RvcmFnZSBNYW5hZ2VyICYgSW5kZXggLS0+CiAgPGcgZmlsdGVyPSJ1cmwoI2Ryb3BT
aGFkb3cpIj4KICAgIDxyZWN0IHg9IjcwMCIgeT0iNTAiIHdpZHRoPSIxNzAiIGhlaWdodD0iMTEw
IiByeD0iOCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFj
aykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIvPgogICAgPHRleHQg
eD0iNzg1IiB5PSI4NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNiIgZm9udC13
ZWlnaHQ9ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPlN0b3JhZ2UgYW5kIEluZGljZXM8L3Rl
eHQ+CiAgICA8dGV4dCB4PSI3NzUiIHk9IjExNSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1z
aXplPSIxNCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+Um93IEhlYXAgLyBUYWJsZXM8L3RleHQ+CiAg
ICA8dGV4dCB4PSI3ODUiIHk9IjEzNSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIx
NCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+VHJlZSBJbmRpY2VzIChPKGxvZyBOKSk8L3RleHQ+CiAg
PC9nPgo8L3N2Zz4=
" alt="Figure 3: Relational Query Processor Execution Pipeline" />
<figcaption>Figure 3: Relational Query Processor Execution
Pipeline</figcaption>
</figure>

1.  **Frontend Lexer and Parser:** The raw SQL string is converted into
    discrete tokens (keywords, literals, identifiers) and parsed into an
    **Abstract Syntax Tree (AST)** using formal grammar rules.

2.  **Planner and Optimiser:** Converts the AST into a directed acyclic
    graph (DAG) of relational algebra operators. An optimiser rearranges
    operations (e.g., pushing selections down before joins to minimise
    intermediate relation cardinality).

3.  **Execution Engine:** Evaluates the plan against physical tables.
    Many modern engines use the **Volcano Iterator Model**, where each
    operator node implements a standard interface:

    ``` cpp
    struct ExecutionNode {
        virtual void open() = 0;
        virtual std::optional<Row> next() = 0;
        virtual void close() = 0;
        virtual ~ExecutionNode() = default;
    };
    ```

4.  **Storage and Indexing Subsystem:** Manages raw in-memory or on-disk
    byte arrays, handles schema validation, allocates row IDs, and
    maintains search structures.

### 16.4 Indexing Structures: Accelerating Lookups

Consider a table `Users` with $N = 10,000,000$ rows. If we execute:

``` sql
SELECT * FROM Users WHERE id = 842101;
```

Without an auxiliary search structure, the engine has no choice but to
perform a **Sequential Scan** (Full Table Scan):

$$\text{Time Complexity} = O(N)$$

At $10^7$ rows, evaluating memory locations sequentially takes tens to
hundreds of milliseconds. When chained across joins, this $O(N)$ cost
cascades into catastrophic $O(N \times M)$ operational deadlocks.

An **Index** is an auxiliary data structure that maps an attribute’s
value directly to the physical storage location (or row identifier) of
the tuple containing that value (fig. 4).

<figure id="fig:db-index">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA4MTAgMjMwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIyMzAiIHZl
cnNpb249IjEuMSIgaWQ9InN2ZzE1IiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmci
PgogIDxkZWZzPgogICAgPHN0eWxlPgogICAgICA6cm9vdCB7CiAgICAgICAgLS1zdmctYmc6IHZh
cigtLWJnLCAjZmZmZmZmKTsKICAgICAgICAtLXN2Zy1mZzogdmFyKC0tZmcsICMwMDAwMDApOwog
ICAgICAgIC0tc3ZnLXRpbnQtMS1mYWxsYmFjazogI2NjY2NjYzsgLyogMjAlIGJsYWNrICsgODAl
IHdoaXRlICovCiAgICAgICAgLS1zdmctdGludC0yLWZhbGxiYWNrOiAjOTk5OTk5OyAvKiA0MCUg
YmxhY2sgKyA2MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10aW50LTE6IGNvbG9yLW1peChpbiBz
cmdiLCB2YXIoLS1zdmctZmcpIDIwJSwgdmFyKC0tc3ZnLWJnKSA4MCUpOwogICAgICAgIC0tc3Zn
LXRpbnQtMjogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgNDAlLCB2YXIoLS1zdmct
YmcpIDYwJSk7CiAgICAgIH0KICAgICAgdGV4dCAgIHsgZm9udC1mYW1pbHk6ICJJb3NldmthIENo
YXJvbiBNb25vIE1lZGl1bSIsIG1vbm9zcGFjZTsgfQogICAgPC9zdHlsZT4KICA8L2RlZnM+CiAg
PCEtLSBJbmRleCBUcmVlIFZpc3VhbGl6YXRpb24gKExlZnQgU2lkZSkgLS0+CiAgPGcgdHJhbnNm
b3JtPSJ0cmFuc2xhdGUoNDAsIDIwKSIgaWQ9Imc1Ij4KICAgIDx0ZXh0IHg9IjE4MCIgeT0iMjAi
IGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTYiIGZvbnQtd2VpZ2h0PSJib2xkIiB0
ZXh0LWFuY2hvcj0ibWlkZGxlIiBpZD0idGV4dDEiPkJTVCAvIEItVHJlZSBJbmRleCAoS2V5OiBV
c2VyIElEKTwvdGV4dD4KICAgIDwhLS0gUm9vdCBOb2RlICg1MCkgLS0+CiAgICA8cmVjdCB4PSIx
NDAiIHk9IjQwIiB3aWR0aD0iODAiIGhlaWdodD0iMzAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmct
dGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZn
KSIgc3Ryb2tlLXdpZHRoPSIyIiBpZD0icmVjdDIiIC8+CiAgICA8dGV4dCB4PSIxODAiIHk9IjYw
IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiB0ZXh0LWFuY2hvcj0ibWlkZGxl
IiBpZD0idGV4dDIiPktleTogNTA8L3RleHQ+CiAgICA8IS0tIEJyYW5jaCBMZWZ0ICgyMCkgJiBS
aWdodCAoNzUpIC0tPgogICAgPGxpbmUgeDE9IjE1NSIgeTE9IjcwIiB4Mj0iOTAiIHkyPSIxMDUi
IHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyIiBpZD0ibGluZTIiIC8+CiAg
ICA8bGluZSB4MT0iMjA1IiB5MT0iNzAiIHgyPSIyNzAiIHkyPSIxMDUiIHN0cm9rZT0idmFyKC0t
c3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyIiBpZD0ibGluZTMiIC8+CiAgICA8cmVjdCB4PSI1MCIg
eT0iMTA1IiB3aWR0aD0iODAiIGhlaWdodD0iMzAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGlu
dC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIg
c3Ryb2tlLXdpZHRoPSIxLjUiIGlkPSJyZWN0MyIgLz4KICAgIDx0ZXh0IHg9IjkwIiB5PSIxMjUi
IGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIHRleHQtYW5jaG9yPSJtaWRkbGUi
IGlkPSJ0ZXh0MyI+S2V5OiAyMDwvdGV4dD4KICAgIDxyZWN0IHg9IjIzMCIgeT0iMTA1IiB3aWR0
aD0iODAiIGhlaWdodD0iMzAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1z
dmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRo
PSIyIiBpZD0icmVjdDQiIC8+CiAgICA8dGV4dCB4PSIyNzAiIHk9IjEyNSIgZmlsbD0idmFyKC0t
c3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIHRleHQtYW5jaG9yPSJt
aWRkbGUiIGlkPSJ0ZXh0NCI+S2V5OiA3NTwvdGV4dD4KICAgIDwhLS0gQnJhbmNoIFJpZ2h0LVJp
Z2h0ICg5MCkgLS0+CiAgICA8bGluZSB4MT0iMjkwIiB5MT0iMTM1IiB4Mj0iMzMwIiB5Mj0iMTcw
IiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIgaWQ9ImxpbmU0IiAvPgog
ICAgPHJlY3QgeD0iMjkwIiB5PSIxNzAiIHdpZHRoPSI4MCIgaGVpZ2h0PSIzMCIgcng9IjYiIGZp
bGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tl
PSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiIGlkPSJyZWN0NSIgLz4KICAgIDx0ZXh0
IHg9IjMzMCIgeT0iMTkwIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBmb250
LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSIgaWQ9InRleHQ1Ij5LZXk6IDkwPC90
ZXh0PgogIDwvZz4KICA8IS0tIENvbm5lY3Rpb24gUG9pbnRlciBmcm9tIEtleSA5MCB0byBIZWFw
IFJvdyAyIC0tPgogIDxwYXRoIGQ9Ik0gNDEwIDIwNSBDIDQ3MCAyMDUsIDQ4MCAxNDUsIDUyMCAx
NDUiIGZpbGw9Im5vbmUiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyLjUi
IHN0cm9rZS1kYXNoYXJyYXk9IjYsNCIgaWQ9InBhdGg1IiAvPgogIDxwb2x5Z29uIHBvaW50cz0i
NTIwLDE0NSA1MTAsMTQwIDUxMCwxNTAiIGZpbGw9InZhcigtLXN2Zy1mZykiIGlkPSJwb2x5Z29u
NSIgLz4KICA8dGV4dCB4PSI0NTUiIHk9IjEzNSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1z
aXplPSIxNCIgdGV4dC1hbmNob3I9Im1pZGRsZSIgaWQ9InRleHQ2Ij5EaXJlY3QgUm93IFBvaW50
ZXI8L3RleHQ+CiAgPCEtLSBQaHlzaWNhbCBEYXRhIFRhYmxlIC8gUm93IFN0b3JhZ2UgKFJpZ2h0
IFNpZGUpIC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDUzMCwgMzUpIiBpZD0iZzE1Ij4K
ICAgIDx0ZXh0IHg9IjE0MCIgeT0iMTUiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0i
MTYiIGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIiBpZD0idGV4dDciPlRh
YmxlIFN0b3JhZ2UgKFJvdyBIZWFwKTwvdGV4dD4KICAgIDwhLS0gUm93IDAgLS0+CiAgICA8cmVj
dCB4PSIwIiB5PSIzMCIgd2lkdGg9IjI4MCIgaGVpZ2h0PSIzMCIgZmlsbD0idmFyKC0tc3ZnLXRp
bnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZyki
IGlkPSJyZWN0NyIgLz4KICAgIDx0ZXh0IHg9IjE1IiB5PSI1MCIgZmlsbD0idmFyKC0tc3ZnLWZn
KSIgZm9udC1zaXplPSIxNCIgaWQ9InRleHQ4Ij5Sb3dJRDogMDwvdGV4dD4KICAgIDx0ZXh0IHg9
IjEwMCIgeT0iNTAiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGlkPSJ0ZXh0
OSI+MjAgfCAmcXVvdDtBZGEmcXVvdDsmI3hhMDsmI3hhMDsmI3hhMDsmI3hhMDt8IDQ1MDAuMDwv
dGV4dD4KICAgIDwhLS0gUm93IDEgLS0+CiAgICA8cmVjdCB4PSIwIiB5PSI2NSIgd2lkdGg9IjI4
MCIgaGVpZ2h0PSIzMCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1m
YWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIGlkPSJyZWN0OSIgLz4KICAgIDx0ZXh0
IHg9IjE1IiB5PSI4NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgaWQ9InRl
eHQxMCI+Um93SUQ6IDE8L3RleHQ+CiAgICA8dGV4dCB4PSIxMDAiIHk9Ijg1IiBmaWxsPSJ2YXIo
LS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBpZD0idGV4dDExIj41MCB8ICZxdW90O0FsYW4mcXVv
dDsmI3hhMDsmI3hhMDsmI3hhMDt8IDUyMDAuMDwvdGV4dD4KICAgIDwhLS0gUm93IDIgKFRhcmdl
dCkgLS0+CiAgICA8cmVjdCB4PSIwIiB5PSIxMDAiIHdpZHRoPSIyODAiIGhlaWdodD0iMzAiIGZp
bGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tl
PSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiIGlkPSJyZWN0MTEiIC8+CiAgICA8dGV4
dCB4PSIxNSIgeT0iMTIwIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBmb250
LXdlaWdodD0iYm9sZCIgaWQ9InRleHQxMiI+Um93SUQ6IDI8L3RleHQ+CiAgICA8dGV4dCB4PSIx
MDAiIHk9IjEyMCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWln
aHQ9ImJvbGQiIGlkPSJ0ZXh0MTMiPjkwIHwgJnF1b3Q7R3JhY2UmcXVvdDsmI3hhMDsmI3hhMDt8
IDYxMDAuMDwvdGV4dD4KICAgIDwhLS0gUm93IDMgLS0+CiAgICA8cmVjdCB4PSIwIiB5PSIxMzUi
IHdpZHRoPSIyODAiIGhlaWdodD0iMzAiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2
Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBpZD0icmVjdDEzIiAv
PgogICAgPHRleHQgeD0iMTUiIHk9IjE1NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXpl
PSIxNCIgaWQ9InRleHQxNCI+Um93SUQ6IDM8L3RleHQ+CiAgICA8dGV4dCB4PSIxMDAiIHk9IjE1
NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgaWQ9InRleHQxNSI+NzUgfCAm
cXVvdDtFZHNnZXImcXVvdDsmI3hhMDt8IDQ4MDAuMDwvdGV4dD4KICA8L2c+Cjwvc3ZnPg==
" alt="Figure 4: BST/B-Tree Index" />
<figcaption>Figure 4: BST/B-Tree Index</figcaption>
</figure>

By building a **Balanced Binary Search Tree** or **B-Tree** index over
the attribute `id`, lookup time drops exponentially:

$$\text{Time Complexity} = O(\log N)$$

For $10,000,000$ rows, $\log_2(10^7) \approx 24$ comparisons. An
operation that previously took $10,000,000$ checks now completes in
roughly two dozen pointer traversals.

### 16.5 Transactions and the ACID Guarantees

Databases operate under hostile conditions: power rails can drop, memory
can become exhausted, assertions can fail, and concurrent threads can
attempt to mutate the exact same record simultaneously. To maintain
operational correctness, database transactions must uphold four
invariant guarantees, formally known as **ACID**:

1.  **Atomicity:** The “all-or-nothing” guarantee. A transaction
    consists of a discrete sequence of operations. If every operation
    succeeds, the transaction commits. If a single constraint check,
    system call, or arithmetic assertion fails, the entire transaction
    is **aborted** and **rolled back**. The database state must appear
    as though the transaction never began.
2.  **Consistency:** A transaction may only transition the database from
    one valid state to another valid state. All declared
    constraints—such as `NOT NULL`, `UNIQUE`, and `PRIMARY KEY`—must
    hold invariant at the point of commit.
3.  **Isolation:** Concurrent transactions must execute without leaking
    intermediate states to one another. The observable result of
    executing multiple transactions concurrently must be identical to
    running them in some strict sequential order (**Serialisability**).
4.  **Durability:** Once a transaction commits, its mutations are
    permanent. Even if the hardware loses power immediately following a
    commit acknowledgment, the state changes persist (historically
    achieved via non-volatile **Write-Ahead Logging (WAL)**).

In our modern C++ engine implementation, we will explicitly enforce
**Atomicity** and **Consistency** through an in-memory **Undo-Log /
Rollback Journal** coupled with strict constraint validators.

### 16.6 The In-Memory Architecture in Modern C++

Let us construct our relational database engine from first principles in
modern C++20.

We will organise our system into four cohesive layers (as also outlined
in fig. 5):

1.  **Algebraic Type Representation (`Value`):** A type-safe,
    variant-based storage model with strong comparison semantics.
2.  **Schema and Table Structure (`Schema`, `Table`):** Value-semantic
    storage with primary key and constraint enforcement.
3.  **Binary Search Tree Index (`Index`):** An associative $O(\log N)$
    search structure mapping column values to internal `RowID` sets.
4.  **SQL Frontend (`Lexer`, `Parser`, `QueryEngine`):** A full
    recursive-descent parser executing mini-SQL queries over our
    relational algebra primitives.

<figure id="fig:statement-to-result">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzNjhwdCIgaGVpZ2h0PSIzMzRwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzNjguMDAgMzM0LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDMz
MCkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gc3FsIC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNzPSJu
b2RlIj4KPHRpdGxlPnNxbDwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxh
Y2siIHBvaW50cz0iMzYwLC0zMjYgMCwtMzI2IDAsLTI5MCAzNjAsLTI5MCAzNjAsLTMyNiIvPgo8
dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxODAiIHk9Ii0zMTEiIGZvbnQtZmFtaWx5PSJJ
b3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+U1FMIFN0YXRlbWVu
dDwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTgwIiB5PSItMzAwIiBmb250
LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPiZx
dW90O1NFTEVDVCBuYW1lLCBzYWxhcnkgRlJPTSBlbmdpbmVlcnMgV0hFUkUgYWdlICZndDs9IDIx
OyZxdW90OzwvdGV4dD4KPC9nPgo8IS0tIGxleGVyIC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJu
b2RlIj4KPHRpdGxlPmxleGVyPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJi
bGFjayIgcG9pbnRzPSIzNjAsLTI2OCAwLC0yNjggMCwtMjMyIDM2MCwtMjMyIDM2MCwtMjY4Ii8+
Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjE4MCIgeT0iLTI1MyIgZm9udC1mYW1pbHk9
Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5MZXhlciAoVG9r
ZW4gU3RyZWFtKTwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTgwIiB5PSIt
MjQyIiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0i
MTAuMDAiPltTRUxFQ1QsIElERU5UKCZxdW90O25hbWUmcXVvdDspLCBDT01NQSwgSURFTlQoJnF1
b3Q7c2FsYXJ5JnF1b3Q7KSwgLi4uXTwvdGV4dD4KPC9nPgo8IS0tIHNxbCYjNDU7Jmd0O2xleGVy
IC0tPgo8ZyBpZD0iZWRnZTEiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPnNxbCYjNDU7Jmd0O2xleGVy
PC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTE4MCwtMjg5Ljg5
QzE4MCwtMjg2LjIgMTgwLC0yODIuMjEgMTgwLC0yNzguMjQiLz4KPHBvbHlnb24gZmlsbD0iYmxh
Y2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTgzLjUsLTI3OCAxODAsLTI2OCAxNzYuNSwtMjc4
IDE4My41LC0yNzgiLz4KPC9nPgo8IS0tIHBhcnNlciAtLT4KPGcgaWQ9Im5vZGUzIiBjbGFzcz0i
bm9kZSI+Cjx0aXRsZT5wYXJzZXI8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9
ImJsYWNrIiBwb2ludHM9IjM2MCwtMjEwIDAsLTIxMCAwLC0xNzQgMzYwLC0xNzQgMzYwLC0yMTAi
Lz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTgwIiB5PSItMTk1IiBmb250LWZhbWls
eT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPlJlY3Vyc2l2
ZSBEZXNjZW50IFNRTCBQYXJzZXI8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9
IjE4MCIgeT0iLTE4NCIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBm
b250LXNpemU9IjEwLjAwIj5BU1QgTm9kZTogU2VsZWN0U3RhdGVtZW50IEFTVCBOb2RlPC90ZXh0
Pgo8L2c+CjwhLS0gbGV4ZXImIzQ1OyZndDtwYXJzZXIgLS0+CjxnIGlkPSJlZGdlMiIgY2xhc3M9
ImVkZ2UiPgo8dGl0bGU+bGV4ZXImIzQ1OyZndDtwYXJzZXI8L3RpdGxlPgo8cGF0aCBmaWxsPSJu
b25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTgwLC0yMzEuODlDMTgwLC0yMjguMiAxODAsLTIyNC4y
MSAxODAsLTIyMC4yNCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9p
bnRzPSIxODMuNSwtMjIwIDE4MCwtMjEwIDE3Ni41LC0yMjAgMTgzLjUsLTIyMCIvPgo8L2c+Cjwh
LS0gcGxhbm5lciAtLT4KPGcgaWQ9Im5vZGU0IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5wbGFubmVy
PC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIzNjAs
LTE1MiAwLC0xNTIgMCwtMTE2IDM2MCwtMTE2IDM2MCwtMTUyIi8+Cjx0ZXh0IHRleHQtYW5jaG9y
PSJtaWRkbGUiIHg9IjE4MCIgeT0iLTE0Mi41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24g
TW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPlF1ZXJ5IFBsYW5uZXI8L3RleHQ+Cjx0ZXh0
IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iOCIgeT0iLTEzMS41IiBmb250LWZhbWlseT0iSW9zZXZr
YSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPkluZGV4IExvb2t1cCBBdmFp
bGFibGU/ICYjNDU7JiM0NTsmZ3Q7IEJTVCBSYW5nZSBRdWVyeTwvdGV4dD4KPHRleHQgdGV4dC1h
bmNob3I9InN0YXJ0IiB4PSI4IiB5PSItMTIwLjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJv
biBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+RWxzZSAmIzE2MDsmIzE2MDsmIzE2MDsm
IzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2
MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzQ1OyYjNDU7Jmd0OyBGdWxs
IFRhYmxlIFNlcXVlbnRpYWwgU2NhbjwvdGV4dD4KPC9nPgo8IS0tIHBhcnNlciYjNDU7Jmd0O3Bs
YW5uZXIgLS0+CjxnIGlkPSJlZGdlMyIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+cGFyc2VyJiM0NTsm
Z3Q7cGxhbm5lcjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0x
ODAsLTE3My44OUMxODAsLTE3MC4yIDE4MCwtMTY2LjIxIDE4MCwtMTYyLjI0Ii8+Cjxwb2x5Z29u
IGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjE4My41LC0xNjIgMTgwLC0xNTIg
MTc2LjUsLTE2MiAxODMuNSwtMTYyIi8+CjwvZz4KPCEtLSBub2RlcyAtLT4KPGcgaWQ9Im5vZGU1
IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5ub2RlczwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUi
IHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMzYwLC05NCAwLC05NCAwLC01OCAzNjAsLTU4IDM2MCwt
OTQiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTgwIiB5PSItNzkiIGZvbnQtZmFt
aWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+RXhlY3V0
aW9uIE5vZGVzPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxODAiIHk9Ii02
OCIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEw
LjAwIj5TZWxlY3Rpb25Ob2RlICjPgykgJiM0NTsmIzQ1OyZndDsgUHJvamVjdGlvbk5vZGUgKM+A
KTwvdGV4dD4KPC9nPgo8IS0tIHBsYW5uZXImIzQ1OyZndDtub2RlcyAtLT4KPGcgaWQ9ImVkZ2U0
IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5wbGFubmVyJiM0NTsmZ3Q7bm9kZXM8L3RpdGxlPgo8cGF0
aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTgwLC0xMTUuODlDMTgwLC0xMTIuMiAx
ODAsLTEwOC4yMSAxODAsLTEwNC4yNCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJi
bGFjayIgcG9pbnRzPSIxODMuNSwtMTA0IDE4MCwtOTQgMTc2LjUsLTEwNCAxODMuNSwtMTA0Ii8+
CjwvZz4KPCEtLSByZXN1bHQgLS0+CjxnIGlkPSJub2RlNiIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+
cmVzdWx0PC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRz
PSIzNjAsLTM2IDAsLTM2IDAsMCAzNjAsMCAzNjAsLTM2Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJt
aWRkbGUiIHg9IjE4MCIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25v
IE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+UmVzdWx0IFRhYmxlPC90ZXh0Pgo8L2c+CjwhLS0g
bm9kZXMmIzQ1OyZndDtyZXN1bHQgLS0+CjxnIGlkPSJlZGdlNSIgY2xhc3M9ImVkZ2UiPgo8dGl0
bGU+bm9kZXMmIzQ1OyZndDtyZXN1bHQ8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9
ImJsYWNrIiBkPSJNMTgwLC01Ny44OUMxODAsLTU0LjIgMTgwLC01MC4yMSAxODAsLTQ2LjI0Ii8+
Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjE4My41LC00NiAx
ODAsLTM2IDE3Ni41LC00NiAxODMuNSwtNDYiLz4KPC9nPgo8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 5: Layers of SQL Statement Processing</figcaption>
</figure>

- lst. 1 defines the core data model. Rather than untyped raw memory
  blocks, we use `std::variant` to represent values cleanly and safely.
- lst. 2 implements the Binary Search Tree Index; an index accelerates
  lookups from an attribute value to matching `RowID` instances. The
  template-free balanced binary tree indexing structure handles `Value`
  keys directly.
- lst. 3 creates the `Table` class, which holds the row heap, executes
  constraint validation, maintains indices, and tracks an **Undo Log**
  for atomic rollback.
- lst. 4 defines a lexical analyser (lexer) to be used with a
  recursive-descent parser, supporting the following grammar:
  `plaintext   Statement     := CreateTable | InsertInto | SelectStmt   CreateTable   := "CREATE" "TABLE" Ident "(" ColumnDefList ")"   ColumnDefList := ColumnDef ( "," ColumnDef )*   ColumnDef     := Ident Type [ "PRIMARY" "KEY" ] [ "NOT" "NULL" ]   InsertInto    := "INSERT" "INTO" Ident "VALUES" "(" LiteralList ")"   SelectStmt    := "SELECT" ProjectionList "FROM" Ident [ "WHERE" Predicate ]   Predicate     := Ident BinaryOp Literal   BinaryOp      := "=" | "!=" | "<" | "<=" | ">" | ">="`
- lst. 5 defines the relational algebra nodes: `PhysicalScanNode` and
  `ProjectionNode` ($\pi$).
- lst. 6 constructs an end-to-end driver showing a test database in
  action:
  1.  Creating relational schemas with constraints.
  2.  Indexing attributes and verifying lookups.
  3.  Executing relational queries via SQL.
  4.  Testing transactional atomicity with rollback on constraint
      violations.

Listing 1: Core Implementation: Type Safety and Schemas (CoreDB.hpp)

In [ ]:
#ifndef COREDB_HPP
#define COREDB_HPP

#include <iostream>
#include <string>
#include <vector>
#include <variant>
#include <optional>
#include <memory>
#include <stdexcept>
#include <functional>
#include <sstream>
#include <map>
#include <algorithm>
#include <cstdint>
#include <string_view>
#include <iomanip>

namespace CoreDB {

// -----------------------------------------------------------------------------
// 1. Primitive Value System
// -----------------------------------------------------------------------------

enum class DataType {
    Integer,
    Float,
    Text,
    Null
};

// Represents a concrete relational value
class Value {
public:
    using VariantType = std::variant<std::monostate, int64_t, double, std::string>;

    Value() : m_data(std::monostate{}) {}
    Value(int64_t v) : m_data(v) {}
    Value(int v) : m_data(static_cast<int64_t>(v)) {}
    Value(double v) : m_data(v) {}
    Value(std::string v) : m_data(std::move(v)) {}
    Value(const char* v) : m_data(std::string(v)) {}

    [[nodiscard]] DataType type() const noexcept {
        return std::visit([](const auto& val) -> DataType {
            using T = std::decay_t<decltype(val)>;
            if constexpr (std::is_same_v<T, std::monostate>) return DataType::Null;
            else if constexpr (std::is_same_v<T, int64_t>)    return DataType::Integer;
            else if constexpr (std::is_same_v<T, double>)     return DataType::Float;
            else if constexpr (std::is_same_v<T, std::string>)return DataType::Text;
        }, m_data);
    }

    [[nodiscard]] bool is_null() const noexcept {
        return std::holds_alternative<std::monostate>(m_data);
    }

    [[nodiscard]] int64_t as_int() const {
        if (auto p = std::get_if<int64_t>(&m_data)) return *p;
        throw std::runtime_error("Type error: Value is not an Integer.");
    }

    [[nodiscard]] double as_float() const {
        if (auto p = std::get_if<double>(&m_data)) return *p;
        if (auto p = std::get_if<int64_t>(&m_data)) return static_cast<double>(*p);
        throw std::runtime_error("Type error: Value is not numeric.");
    }

    [[nodiscard]] const std::string& as_text() const {
        if (auto p = std::get_if<std::string>(&m_data)) return *p;
        throw std::runtime_error("Type error: Value is not Text.");
    }

    // Equality operator
    bool operator==(const Value& other) const {
        if (type() != other.type()) {
            // Support transparent float-to-int comparison
            if ((type() == DataType::Integer && other.type() == DataType::Float) ||
                (type() == DataType::Float && other.type() == DataType::Integer)) {
                return as_float() == other.as_float();
            }
            return false;
        }
        return m_data == other.m_data;
    }

    // Ordering operator for BST indexing and sorting
    bool operator<(const Value& other) const {
        if (type() != other.type()) {
            if ((type() == DataType::Integer && other.type() == DataType::Float) ||
                (type() == DataType::Float && other.type() == DataType::Integer)) {
                return as_float() < other.as_float();
            }
            return type() < other.type();
        }
        return m_data < other.m_data;
    }

    bool operator<=(const Value& other) const { return *this < other || *this == other; }
    bool operator>(const Value& other) const { return !(*this <= other); }
    bool operator>=(const Value& other) const { return !(*this < other); }
    bool operator!=(const Value& other) const { return !(*this == other); }

    friend std::ostream& operator<<(std::ostream& os, const Value& val) {
        std::visit([&os](const auto& v) {
            using T = std::decay_t<decltype(v)>;
            if constexpr (std::is_same_v<T, std::monostate>) {
                os << "NULL";
            } else if constexpr (std::is_same_v<T, std::string>) {
                os << "'" << v << "'";
            } else {
                os << v;
            }
        }, val.m_data);
        return os;
    }

private:
    VariantType m_data;
};

// -----------------------------------------------------------------------------
// 2. Schema Definition and Constraints
// -----------------------------------------------------------------------------

struct Column {
    std::string name;
    DataType type;
    bool is_primary_key{false};
    bool is_not_null{false};
};

class Schema {
public:
    Schema() = default;
    Schema(std::initializer_list<Column> cols) : m_columns(cols) {}
    explicit Schema(std::vector<Column> cols) : m_columns(std::move(cols)) {}

    [[nodiscard]] const std::vector<Column>& columns() const noexcept { return m_columns; }

    [[nodiscard]] std::optional<size_t> index_of(std::string_view column_name) const noexcept {
        for (size_t i = 0; i < m_columns.size(); ++i) {
            if (m_columns[i].name == column_name) return i;
        }
        return std::nullopt;
    }

    [[nodiscard]] const Column& get_column(size_t idx) const {
        if (idx >= m_columns.size()) throw std::out_of_range("Column index out of bounds.");
        return m_columns[idx];
    }

    [[nodiscard]] size_t arity() const noexcept { return m_columns.size(); }

private:
    std::vector<Column> m_columns;
};

using Row = std::vector<Value>;
using RowID = size_t;

} // namespace CoreDB

#endif // COREDB_HPP

Listing 2: Binary Search Tree Index (BSTIndex.hpp)

In [ ]:
#ifndef BST_INDEX_HPP
#define BST_INDEX_HPP

// #include "CoreDB.hpp"

namespace CoreDB {

class BSTIndex {
private:
    struct Node {
        Value key;
        std::vector<RowID> row_ids;
        std::unique_ptr<Node> left;
        std::unique_ptr<Node> right;
        int height{1};

        explicit Node(Value k, RowID id) : key(std::move(k)) {
            row_ids.push_back(id);
        }
    };

    std::unique_ptr<Node> m_root;

    int get_height(const Node* n) const noexcept {
        return n ? n->height : 0;
    }

    int balance_factor(const Node* n) const noexcept {
        return n ? get_height(n->left.get()) - get_height(n->right.get()) : 0;
    }

    void update_height(Node* n) noexcept {
        if (n) {
            n->height = 1 + std::max(get_height(n->left.get()), get_height(n->right.get()));
        }
    }

    std::unique_ptr<Node> rotate_right(std::unique_ptr<Node> y) {
        auto x = std::move(y->left);
        y->left = std::move(x->right);
        update_height(y.get());
        x->right = std::move(y);
        update_height(x.get());
        return x;
    }

    std::unique_ptr<Node> rotate_left(std::unique_ptr<Node> x) {
        auto y = std::move(x->right);
        x->right = std::move(y->left);
        update_height(x.get());
        y->left = std::move(x);
        update_height(y.get());
        return y;
    }

    std::unique_ptr<Node> insert_impl(std::unique_ptr<Node> node, const Value& key, RowID id) {
        if (!node) {
            return std::make_unique<Node>(key, id);
        }

        if (key == node->key) {
            node->row_ids.push_back(id);
            return node;
        }

        if (key < node->key) {
            node->left = insert_impl(std::move(node->left), key, id);
        } else {
            node->right = insert_impl(std::move(node->right), key, id);
        }

        update_height(node.get());
        int balance = balance_factor(node.get());

        // AVL Rebalancing
        if (balance > 1 && key < node->left->key) {
            return rotate_right(std::move(node));
        }
        if (balance < -1 && key > node->right->key) {
            return rotate_left(std::move(node));
        }
        if (balance > 1 && key > node->left->key) {
            node->left = rotate_left(std::move(node->left));
            return rotate_right(std::move(node));
        }
        if (balance < -1 && key < node->right->key) {
            node->right = rotate_right(std::move(node->right));
            return rotate_left(std::move(node));
        }

        return node;
    }

    void find_range_impl(const Node* node, const Value& low, const Value& high, 
                         std::vector<RowID>& results) const {
        if (!node) return;

        if (low < node->key) {
            find_range_impl(node->left.get(), low, high, results);
        }
        if (node->key >= low && node->key <= high) {
            results.insert(results.end(), node->row_ids.begin(), node->row_ids.end());
        }
        if (high > node->key) {
            find_range_impl(node->right.get(), low, high, results);
        }
    }

public:
    BSTIndex() = default;

    void insert(const Value& key, RowID id) {
        m_root = insert_impl(std::move(m_root), key, id);
    }

    [[nodiscard]] std::vector<RowID> find_exact(const Value& key) const {
        const Node* curr = m_root.get();
        while (curr) {
            if (key == curr->key) {
                return curr->row_ids;
            }
            if (key < curr->key) {
                curr = curr->left.get();
            } else {
                curr = curr->right.get();
            }
        }
        return {};
    }

    [[nodiscard]] std::vector<RowID> find_range(const Value& low, const Value& high) const {
        std::vector<RowID> results;
        find_range_impl(m_root.get(), low, high, results);
        return results;
    }
};

} // namespace CoreDB

#endif // BST_INDEX_HPP

Listing 3: Transactional Table Storage with Atomic Rollback (Table.hpp)

In [ ]:
#ifndef TABLE_HPP
#define TABLE_HPP

// #include "CoreDB.hpp"
// #include "BSTIndex.hpp"

namespace CoreDB {

class Table {
public:
    Table(std::string name, Schema schema)
        : m_name(std::move(name)), m_schema(std::move(schema)) {
        // Automatically index the primary key column
        for (size_t i = 0; i < m_schema.arity(); ++i) {
            if (m_schema.get_column(i).is_primary_key) {
                m_pk_index_col = i;
                m_indices[i] = std::make_unique<BSTIndex>();
            }
        }
    }

    [[nodiscard]] const std::string& name() const noexcept { return m_name; }
    [[nodiscard]] const Schema& schema() const noexcept { return m_schema; }
    [[nodiscard]] size_t size() const noexcept { return m_active_row_count; }

    void create_index_on(const std::string& column_name) {
        auto col_idx = m_schema.index_of(column_name);
        if (!col_idx) {
            throw std::runtime_error("Column '" + column_name + "' does not exist.");
        }
        size_t idx = *col_idx;
        if (m_indices.contains(idx)) return; // Already indexed

        auto new_index = std::make_unique<BSTIndex>();
        for (size_t r = 0; r < m_rows.size(); ++r) {
            if (m_row_active[r]) {
                new_index->insert(m_rows[r][idx], r);
            }
        }
        m_indices[idx] = std::move(new_index);
    }

    [[nodiscard]] const BSTIndex* get_index(size_t col_idx) const {
        auto it = m_indices.find(col_idx);
        if (it != m_indices.end()) return it->second.get();
        return nullptr;
    }

    // -------------------------------------------------------------------------
    // Transactional Mutations
    // -------------------------------------------------------------------------

    void begin_transaction() {
        m_in_transaction = true;
        m_undo_log.clear();
    }

    void commit() {
        if (!m_in_transaction) return;
        m_undo_log.clear();
        m_in_transaction = false;
    }

    void rollback() {
        if (!m_in_transaction) return;
        
        // Revert mutations in reverse order
        for (auto it = m_undo_log.rbegin(); it != m_undo_log.rend(); ++it) {
            RowID id = it->row_id;
            if (it->type == UndoType::Insert) {
                m_row_active[id] = false;
                --m_active_row_count;
            }
        }
        
        // Rebuild indices to reflect restored state
        for (auto& [col_idx, index_ptr] : m_indices) {
            auto fresh_index = std::make_unique<BSTIndex>();
            for (size_t r = 0; r < m_rows.size(); ++r) {
                if (m_row_active[r]) {
                    fresh_index->insert(m_rows[r][col_idx], r);
                }
            }
            index_ptr = std::move(fresh_index);
        }

        m_undo_log.clear();
        m_in_transaction = false;
    }

    void insert_record(Row row) {
        validate_row(row);

        // Check primary key uniqueness
        if (m_pk_index_col.has_value()) {
            size_t pk_col = *m_pk_index_col;
            const Value& pk_val = row[pk_col];
            auto existing = m_indices[pk_col]->find_exact(pk_val);
            if (!existing.empty()) {
                throw std::runtime_error("Constraint Violation: Duplicate Primary Key value " + 
                                         [pk_val]() { 
                                             std::ostringstream ss; 
                                             ss << pk_val; 
                                             return ss.str(); 
                                         }());
            }
        }

        RowID new_id = m_rows.size();
        m_rows.push_back(std::move(row));
        m_row_active.push_back(true);
        ++m_active_row_count;

        // Update all associated indices
        for (auto& [col_idx, index_ptr] : m_indices) {
            index_ptr->insert(m_rows[new_id][col_idx], new_id);
        }

        if (m_in_transaction) {
            m_undo_log.push_back({UndoType::Insert, new_id});
        }
    }

    [[nodiscard]] const Row& get_row(RowID id) const {
        if (id >= m_rows.size() || !m_row_active[id]) {
            throw std::out_of_range("Invalid RowID.");
        }
        return m_rows[id];
    }

    [[nodiscard]] std::vector<RowID> all_row_ids() const {
        std::vector<RowID> ids;
        ids.reserve(m_active_row_count);
        for (size_t i = 0; i < m_rows.size(); ++i) {
            if (m_row_active[i]) ids.push_back(i);
        }
        return ids;
    }

private:
    enum class UndoType { Insert };
    struct UndoEntry {
        UndoType type;
        RowID row_id;
    };

    std::string m_name;
    Schema m_schema;
    std::vector<Row> m_rows;
    std::vector<bool> m_row_active;
    size_t m_active_row_count{0};

    std::optional<size_t> m_pk_index_col;
    std::map<size_t, std::unique_ptr<BSTIndex>> m_indices;

    bool m_in_transaction{false};
    std::vector<UndoEntry> m_undo_log;

    void validate_row(const Row& row) const {
        if (row.size() != m_schema.arity()) {
            throw std::runtime_error("Schema Mismatch: Expected " + 
                                     std::to_string(m_schema.arity()) + 
                                     " attributes, got " + std::to_string(row.size()));
        }
        for (size_t i = 0; i < m_schema.arity(); ++i) {
            const auto& col = m_schema.get_column(i);
            const auto& val = row[i];

            if (col.is_not_null && val.is_null()) {
                throw std::runtime_error("Constraint Violation: Column '" + 
                                         col.name + "' cannot be NULL.");
            }
            if (!val.is_null() && val.type() != col.type) {
                // Allow safe int-to-float promotion
                if (col.type == DataType::Float && val.type() == DataType::Integer) {
                    continue;
                }
                throw std::runtime_error("Type Mismatch on Column '" + col.name + "'.");
            }
        }
    }
};

} // namespace CoreDB

#endif // TABLE_HPP

Listing 4: Lexical Tokenisation (Lexer.hpp)

In [ ]:
#ifndef LEXER_HPP
#define LEXER_HPP

// #include "CoreDB.hpp"

namespace CoreDB {

enum class TokenType {
    Keyword_CREATE, Keyword_TABLE, Keyword_INSERT, Keyword_INTO,
    Keyword_VALUES, Keyword_SELECT, Keyword_FROM, Keyword_WHERE,
    Keyword_PRIMARY, Keyword_KEY, Keyword_NOT, Keyword_NULL,
    Type_INT, Type_FLOAT, Type_TEXT,
    Identifier,
    Literal_Integer, Literal_Float, Literal_String,
    Comma, LeftParen, RightParen, Asterisk,
    Op_Equal, Op_NotEqual, Op_Less, Op_LessEqual, Op_Greater, Op_GreaterEqual,
    SemiColon,
    EndOfFile
};

struct Token {
    TokenType type;
    std::string text;
    size_t line{1};
};

class Lexer {
public:
    explicit Lexer(std::string source) : m_src(std::move(source)), m_pos(0) {}

    std::vector<Token> tokenise() {
        std::vector<Token> tokens;
        while (!is_eof()) {
            char c = peek();
            if (std::isspace(c)) {
                advance();
                continue;
            }
            if (std::isalpha(c) || c == '_') {
                tokens.push_back(lex_identifier());
                continue;
            }
            if (std::isdigit(c) || (c == '-' && std::isdigit(peek_next()))) {
                tokens.push_back(lex_number());
                continue;
            }
            if (c == '\'') {
                tokens.push_back(lex_string());
                continue;
            }
            tokens.push_back(lex_symbol());
        }
        tokens.push_back({TokenType::EndOfFile, "", m_line});
        return tokens;
    }

private:
    std::string m_src;
    size_t m_pos{0};
    size_t m_line{1};

    [[nodiscard]] bool is_eof() const noexcept { return m_pos >= m_src.size(); }
    [[nodiscard]] char peek() const noexcept { return is_eof() ? '\0' : m_src[m_pos]; }
    [[nodiscard]] char peek_next() const noexcept { 
        return (m_pos + 1 >= m_src.size()) ? '\0' : m_src[m_pos + 1]; 
    }
    char advance() noexcept { return m_src[m_pos++]; }

    Token lex_identifier() {
        size_t start = m_pos;
        while (!is_eof() && (std::isalnum(peek()) || peek() == '_')) {
            advance();
        }
        std::string raw = m_src.substr(start, m_pos - start);
        std::string upper = raw;
        std::transform(upper.begin(), upper.end(), upper.begin(), ::toupper);

        if (upper == "CREATE")   return {TokenType::Keyword_CREATE, raw, m_line};
        if (upper == "TABLE")    return {TokenType::Keyword_TABLE, raw, m_line};
        if (upper == "INSERT")   return {TokenType::Keyword_INSERT, raw, m_line};
        if (upper == "INTO")     return {TokenType::Keyword_INTO, raw, m_line};
        if (upper == "VALUES")   return {TokenType::Keyword_VALUES, raw, m_line};
        if (upper == "SELECT")   return {TokenType::Keyword_SELECT, raw, m_line};
        if (upper == "FROM")     return {TokenType::Keyword_FROM, raw, m_line};
        if (upper == "WHERE")    return {TokenType::Keyword_WHERE, raw, m_line};
        if (upper == "PRIMARY")  return {TokenType::Keyword_PRIMARY, raw, m_line};
        if (upper == "KEY")      return {TokenType::Keyword_KEY, raw, m_line};
        if (upper == "NOT")      return {TokenType::Keyword_NOT, raw, m_line};
        if (upper == "NULL")     return {TokenType::Keyword_NULL, raw, m_line};
        if (upper == "INT")      return {TokenType::Type_INT, raw, m_line};
        if (upper == "FLOAT")    return {TokenType::Type_FLOAT, raw, m_line};
        if (upper == "TEXT")     return {TokenType::Type_TEXT, raw, m_line};

        return {TokenType::Identifier, raw, m_line};
    }

    Token lex_number() {
        size_t start = m_pos;
        if (peek() == '-') advance();
        bool is_float = false;
        while (!is_eof() && std::isdigit(peek())) advance();
        if (!is_eof() && peek() == '.') {
            is_float = true;
            advance();
            while (!is_eof() && std::isdigit(peek())) advance();
        }
        std::string val = m_src.substr(start, m_pos - start);
        return {is_float ? TokenType::Literal_Float : TokenType::Literal_Integer, val, m_line};
    }

    Token lex_string() {
        advance(); // Skip opening '
        size_t start = m_pos;
        while (!is_eof() && peek() != '\'') {
            advance();
        }
        if (is_eof()) throw std::runtime_error("Lexer error: Unterminated string literal.");
        std::string val = m_src.substr(start, m_pos - start);
        advance(); // Skip closing '
        return {TokenType::Literal_String, val, m_line};
    }

    Token lex_symbol() {
        char c = advance();
        switch (c) {
            case '(': return {TokenType::LeftParen, "(", m_line};
            case ')': return {TokenType::RightParen, ")", m_line};
            case ',': return {TokenType::Comma, ",", m_line};
            case '*': return {TokenType::Asterisk, "*", m_line};
            case '=': return {TokenType::Op_Equal, "=", m_line};
            case '<':
                if (peek() == '=') { advance(); return {TokenType::Op_LessEqual, "<=", m_line}; }
                if (peek() == '>') { advance(); return {TokenType::Op_NotEqual, "<>", m_line}; }
                return {TokenType::Op_Less, "<", m_line};
            case '>':
                if (peek() == '=') { advance(); return {TokenType::Op_GreaterEqual, ">=", m_line}; }
                return {TokenType::Op_Greater, ">", m_line};
            case '!':
                if (peek() == '=') { advance(); return {TokenType::Op_NotEqual, "!=", m_line}; }
                break;
            case ';': return {TokenType::SemiColon, ";", m_line};
            default: break;
        }
        throw std::runtime_error(std::string("Lexer error: Unknown symbol '") + c + "'");
    }
};

} // namespace CoreDB

#endif // LEXER_HPP

Listing 5: Query Engine: Relational Operations (Database.hpp)

In [ ]:
#ifndef DATABASE_HPP
#define DATABASE_HPP

// #include "CoreDB.hpp"
// #include "BSTIndex.hpp"
// #include "Table.hpp"
// #include "Lexer.hpp"

namespace CoreDB {

enum class ComparisonOp {
    Equal,
    NotEqual,
    LessThan,
    LessOrEqual,
    GreaterThan,
    GreaterOrEqual
};

struct Predicate {
    std::string column_name;
    ComparisonOp op;
    Value value;

    [[nodiscard]] bool evaluate(const Value& cell) const {
        switch (op) {
            case ComparisonOp::Equal:          return cell == value;
            case ComparisonOp::NotEqual:       return cell != value;
            case ComparisonOp::LessThan:       return cell < value;
            case ComparisonOp::LessOrEqual:    return cell <= value;
            case ComparisonOp::GreaterThan:    return cell > value;
            case ComparisonOp::GreaterOrEqual: return cell >= value;
        }
        return false;
    }
};

// -----------------------------------------------------------------------------
// Relational Algebra Execution Engine (Volcano Iterator Pattern)
// -----------------------------------------------------------------------------

class ExecutionNode {
public:
    virtual ~ExecutionNode() = default;
    virtual void open() = 0;
    virtual std::optional<Row> next() = 0;
    virtual const Schema& output_schema() const = 0;
};

// 1. Scan Node with Index Acceleration
class PhysicalScanNode : public ExecutionNode {
public:
    PhysicalScanNode(const Table& table, std::optional<Predicate> pred = std::nullopt)
        : m_table(table), m_predicate(std::move(pred)) {}

    void open() override {
        m_cursor = 0;
        m_matching_ids.clear();

        // Check if an index can accelerate this predicate
        if (m_predicate.has_value()) {
            auto col_idx = m_table.schema().index_of(m_predicate->column_name);
            if (col_idx.has_value()) {
                const BSTIndex* index = m_table.get_index(*col_idx);
                if (index && m_predicate->op == ComparisonOp::Equal) {
                    // Logarithmic O(log N) lookup
                    m_matching_ids = index->find_exact(m_predicate->value);
                    m_uses_index = true;
                    return;
                }
            }
        }

        // Fallback: Full Table Scan O(N)
        m_matching_ids = m_table.all_row_ids();
        m_uses_index = false;
    }

    std::optional<Row> next() override {
        while (m_cursor < m_matching_ids.size()) {
            RowID id = m_matching_ids[m_cursor++];
            const Row& candidate = m_table.get_row(id);

            // If we used an exact index, the predicate is guaranteed to match
            if (m_uses_index) {
                return candidate;
            }

            // Otherwise, evaluate the selection predicate manually
            if (!m_predicate.has_value()) {
                return candidate;
            }

            auto col_idx = *m_table.schema().index_of(m_predicate->column_name);
            if (m_predicate->evaluate(candidate[col_idx])) {
                return candidate;
            }
        }
        return std::nullopt;
    }

    const Schema& output_schema() const override {
        return m_table.schema();
    }

    [[nodiscard]] bool used_index() const noexcept { return m_uses_index; }

private:
    const Table& m_table;
    std::optional<Predicate> m_predicate;
    std::vector<RowID> m_matching_ids;
    size_t m_cursor{0};
    bool m_uses_index{false};
};

// 2. Projection Node (π)
class ProjectionNode : public ExecutionNode {
public:
    ProjectionNode(std::unique_ptr<ExecutionNode> child, std::vector<std::string> projected_cols)
        : m_child(std::move(child)), m_requested_names(std::move(projected_cols)) {
        
        const auto& in_schema = m_child->output_schema();
        std::vector<Column> out_cols;

        if (m_requested_names.size() == 1 && m_requested_names[0] == "*") {
            output_m_schema = in_schema;
            for (size_t i = 0; i < in_schema.arity(); ++i) {
                project_m_indices.push_back(i);
            }
        } else {
            for (const auto& name : m_requested_names) {
                auto idx = in_schema.index_of(name);
                if (!idx) throw std::runtime_error("Column '" + name + "' does not exist in child relation.");
                project_m_indices.push_back(*idx);
                out_cols.push_back(in_schema.get_column(*idx));
            }
            output_m_schema = Schema(out_cols);
        }
    }

    void open() override {
        m_child->open();
    }

    std::optional<Row> next() override {
        auto child_row = m_child->next();
        if (!child_row) return std::nullopt;

        Row projected;
        projected.reserve(project_m_indices.size());
        for (size_t idx : project_m_indices) {
            projected.push_back((*child_row)[idx]);
        }
        return projected;
    }

    const Schema& output_schema() const override {
        return output_m_schema;
    }

private:
    std::unique_ptr<ExecutionNode> m_child;
    std::vector<std::string> m_requested_names;
    std::vector<size_t> project_m_indices;
    Schema output_m_schema;
};

// -----------------------------------------------------------------------------
// Database Instance: Catalog and Engine Coordination
// -----------------------------------------------------------------------------

class Database {
public:
    void execute_ddl(std::string_view sql) {
        Lexer lexer{std::string(sql)};
        auto tokens = lexer.tokenise();
        size_t pos = 0;

        if (match(tokens, pos, TokenType::Keyword_CREATE)) {
            consume(tokens, pos, TokenType::Keyword_TABLE, "Expected 'TABLE'");
            Token m_nametok = consume(tokens, pos, TokenType::Identifier, "Expected table name");
            consume(tokens, pos, TokenType::LeftParen, "Expected '('");

            std::vector<Column> cols;
            while (true) {
                Token col_name = consume(tokens, pos, TokenType::Identifier, "Expected column name");
                Token col_type = advance_tok(tokens, pos);
                DataType dtype;
                if (col_type.type == TokenType::Type_INT) dtype = DataType::Integer;
                else if (col_type.type == TokenType::Type_FLOAT) dtype = DataType::Float;
                else if (col_type.type == TokenType::Type_TEXT) dtype = DataType::Text;
                else throw std::runtime_error("Invalid data type for column " + col_name.text);

                bool is_pk = false;
                bool is_not_null = false;

                while (pos < tokens.size() && 
                       (tokens[pos].type == TokenType::Keyword_PRIMARY || 
                        tokens[pos].type == TokenType::Keyword_NOT)) {
                    if (match(tokens, pos, TokenType::Keyword_PRIMARY)) {
                        consume(tokens, pos, TokenType::Keyword_KEY, "Expected 'KEY' after 'PRIMARY'");
                        is_pk = true;
                        is_not_null = true;
                    } else if (match(tokens, pos, TokenType::Keyword_NOT)) {
                        consume(tokens, pos, TokenType::Keyword_NULL, "Expected 'NULL' after 'NOT'");
                        is_not_null = true;
                    }
                }

                cols.push_back({col_name.text, dtype, is_pk, is_not_null});

                if (match(tokens, pos, TokenType::Comma)) continue;
                if (match(tokens, pos, TokenType::RightParen)) break;
                throw std::runtime_error("Expected ',' or ')' in column definitions.");
            }

            m_tables.emplace(m_nametok.text, std::make_unique<Table>(m_nametok.text, Schema(cols)));
        } else {
            throw std::runtime_error("Unsupported DDL command.");
        }
    }

    void execute_insert(std::string_view sql) {
        Lexer lexer{std::string(sql)};
        auto tokens = lexer.tokenise();
        size_t pos = 0;

        consume(tokens, pos, TokenType::Keyword_INSERT, "Expected 'INSERT'");
        consume(tokens, pos, TokenType::Keyword_INTO, "Expected 'INTO'");
        Token tbl_tok = consume(tokens, pos, TokenType::Identifier, "Expected table name");
        consume(tokens, pos, TokenType::Keyword_VALUES, "Expected 'VALUES'");
        consume(tokens, pos, TokenType::LeftParen, "Expected '('");

        auto it = m_tables.find(tbl_tok.text);
        if (it == m_tables.end()) {
            throw std::runtime_error("Table '" + tbl_tok.text + "' does not exist.");
        }

        Row row;
        while (true) {
            Token val_tok = advance_tok(tokens, pos);
            if (val_tok.type == TokenType::Literal_Integer) {
                row.emplace_back(static_cast<int64_t>(std::stoll(val_tok.text)));
            } else if (val_tok.type == TokenType::Literal_Float) {
                row.emplace_back(std::stod(val_tok.text));
            } else if (val_tok.type == TokenType::Literal_String) {
                row.emplace_back(val_tok.text);
            } else if (val_tok.type == TokenType::Keyword_NULL) {
                row.emplace_back(Value());
            } else {
                throw std::runtime_error("Unexpected token in VALUES clause: " + val_tok.text);
            }

            if (match(tokens, pos, TokenType::Comma)) continue;
            if (match(tokens, pos, TokenType::RightParen)) break;
            throw std::runtime_error("Expected ',' or ')' in VALUES clause.");
        }

        it->second->insert_record(std::move(row));
    }

    void print_query(std::string_view sql) {
        Lexer lexer{std::string(sql)};
        auto tokens = lexer.tokenise();
        size_t pos = 0;

        consume(tokens, pos, TokenType::Keyword_SELECT, "Expected 'SELECT'");

        std::vector<std::string> projections;
        if (match(tokens, pos, TokenType::Asterisk)) {
            projections.push_back("*");
        } else {
            while (true) {
                Token col = consume(tokens, pos, TokenType::Identifier, "Expected column name");
                projections.push_back(col.text);
                if (!match(tokens, pos, TokenType::Comma)) break;
            }
        }

        consume(tokens, pos, TokenType::Keyword_FROM, "Expected 'FROM'");
        Token tbl_tok = consume(tokens, pos, TokenType::Identifier, "Expected table name");

        auto it = m_tables.find(tbl_tok.text);
        if (it == m_tables.end()) {
            throw std::runtime_error("Table '" + tbl_tok.text + "' does not exist.");
        }
        Table& table = *(it->second);

        std::optional<Predicate> predicate;
        if (match(tokens, pos, TokenType::Keyword_WHERE)) {
            Token col = consume(tokens, pos, TokenType::Identifier, "Expected column name in WHERE");
            Token op_tok = advance_tok(tokens, pos);
            Token val_tok = advance_tok(tokens, pos);

            ComparisonOp op;
            if (op_tok.type == TokenType::Op_Equal) op = ComparisonOp::Equal;
            else if (op_tok.type == TokenType::Op_NotEqual) op = ComparisonOp::NotEqual;
            else if (op_tok.type == TokenType::Op_Less) op = ComparisonOp::LessThan;
            else if (op_tok.type == TokenType::Op_LessEqual) op = ComparisonOp::LessOrEqual;
            else if (op_tok.type == TokenType::Op_Greater) op = ComparisonOp::GreaterThan;
            else if (op_tok.type == TokenType::Op_GreaterEqual) op = ComparisonOp::GreaterOrEqual;
            else throw std::runtime_error("Unsupported comparison operator: " + op_tok.text);

            Value rhs;
            if (val_tok.type == TokenType::Literal_Integer) rhs = Value(static_cast<int64_t>(std::stoll(val_tok.text)));
            else if (val_tok.type == TokenType::Literal_Float) rhs = Value(std::stod(val_tok.text));
            else if (val_tok.type == TokenType::Literal_String) rhs = Value(val_tok.text);
            else throw std::runtime_error("Invalid literal in WHERE predicate.");

            predicate = Predicate{col.text, op, rhs};
        }

        // Construct Relational Algebra Pipeline
        auto scan_node = std::make_unique<PhysicalScanNode>(table, predicate);
        auto proj_node = std::make_unique<ProjectionNode>(std::move(scan_node), projections);

        // Execute Plan
        proj_node->open();
        const auto& out_schema = proj_node->output_schema();

        // Print Pretty Table Header
        std::cout << "\n+";
        for (const auto& col : out_schema.columns()) {
            std::cout << "----------------------+";
        }
        std::cout << "\n|";
        for (const auto& col : out_schema.columns()) {
            std::cout << " " << std::setw(20) << std::left << col.name << " |";
        }
        std::cout << "\n+";
        for (const auto& col : out_schema.columns()) {
            std::cout << "----------------------+";
        }
        std::cout << "\n";

        size_t row_count = 0;
        while (auto row = proj_node->next()) {
            ++row_count;
            std::cout << "|";
            for (const auto& val : *row) {
                std::ostringstream ss;
                ss << val;
                std::cout << " " << std::setw(20) << std::left << ss.str() << " |";
            }
            std::cout << "\n";
        }

        std::cout << "+";
        for (const auto& col : out_schema.columns()) {
            std::cout << "----------------------+";
        }
        std::cout << "\n(" << row_count << " rows returned)\n\n";
    }

    [[nodiscard]] Table& get_table(const std::string& name) {
        auto it = m_tables.find(name);
        if (it == m_tables.end()) throw std::runtime_error("Table not found: " + name);
        return *(it->second);
    }

private:
    std::map<std::string, std::unique_ptr<Table>> m_tables;

    bool match(const std::vector<Token>& tokens, size_t& pos, TokenType type) {
        if (pos < tokens.size() && tokens[pos].type == type) {
            ++pos;
            return true;
        }
        return false;
    }

    Token advance_tok(const std::vector<Token>& tokens, size_t& pos) {
        if (pos >= tokens.size()) throw std::runtime_error("Unexpected end of tokens.");
        return tokens[pos++];
    }

    Token consume(const std::vector<Token>& tokens, size_t& pos, TokenType expected, std::string_view err) {
        if (pos < tokens.size() && tokens[pos].type == expected) {
            return tokens[pos++];
        }
        throw std::runtime_error(std::string(err) + " at token '" + 
                                 (pos < tokens.size() ? tokens[pos].text : "EOF") + "'");
    }
};

} // namespace CoreDB

#endif // DATABASE_HPP

Listing 6: Verification and Demonstration (main.cpp)

In [ ]:
// #include "Database.hpp"
#include <iostream>

int main() {
    using namespace CoreDB;

    std::cout << "====================================================\n";
    std::cout << "      CoreDB Modern C++ Relational Engine Demo      \n";
    std::cout << "====================================================\n\n";

    try {
        Database db;

        // 1. Create Tables
        std::cout << "[Step 1] Creating schema for 'engineers'...\n";
        db.execute_ddl(
            "CREATE TABLE engineers ("
            "  id INT PRIMARY KEY,"
            "  name TEXT NOT NULL,"
            "  salary FLOAT NOT NULL,"
            "  dept TEXT"
            ");"
        );

        // 2. Insert Records
        std::cout << "[Step 2] Inserting records into 'engineers'...\n";
        db.execute_insert("INSERT INTO engineers VALUES (101, 'Ada Lovelace', 8500.50, 'Systems');");
        db.execute_insert("INSERT INTO engineers VALUES (102, 'Alan Turing', 9200.00, 'Cryptography');");
        db.execute_insert("INSERT INTO engineers VALUES (103, 'Grace Hopper', 8900.00, 'Compilers');");
        db.execute_insert("INSERT INTO engineers VALUES (104, 'Claude Shannon', 9100.00, 'Information Theory');");
        db.execute_insert("INSERT INTO engineers VALUES (105, 'John von Neumann', 9950.00, 'Architecture');");

        // 3. Query All Data (π * σ true)
        std::cout << "[Step 3] Query: SELECT * FROM engineers;";
        db.print_query("SELECT * FROM engineers;");

        // 4. Query with Projection and Selection Filter
        std::cout << "[Step 4] Query: SELECT name, salary FROM engineers WHERE salary > 9000.00;";
        db.print_query("SELECT name, salary FROM engineers WHERE salary > 9000.00;");

        // 5. Index-Accelerated Point Query on Primary Key
        std::cout << "[Step 5] Query: SELECT id, name, dept FROM engineers WHERE id = 103;";
        db.print_query("SELECT id, name, dept FROM engineers WHERE id = 103;");

        // 6. Demonstrate Transaction Atomicity and Constraint Rollback
        std::cout << "[Step 6] Testing Transactional ACID Guarantees...\n";
        Table& eng_table = db.get_table("engineers");

        std::cout << " -> Starting Transaction...\n";
        eng_table.begin_transaction();

        eng_table.insert_record({106, "Barbara Liskov", 9400.0, "Distributed"});
        std::cout << " -> Inserted 'Barbara Liskov' successfully.\n";

        try {
            std::cout << " -> Attempting to insert duplicate Primary Key (id: 101)...\n";
            eng_table.insert_record({101, "Imposter Record", 1000.0, "Security"});
        } catch (const std::exception& e) {
            std::cerr << " -> [Caught Expected Constraint Violation]: " << e.what() << "\n";
            std::cout << " -> Executing Atomic ROLLBACK...\n";
            eng_table.rollback();
        }

        std::cout << "\n[Step 7] Verifying table state post-rollback (ID 106 should NOT exist):";
        db.print_query("SELECT id, name FROM engineers;");

    } catch (const std::exception& e) {
        std::cerr << "CRITICAL ERROR: " << e.what() << "\n";
        return 1;
    }

    return 0;
}

### 16.13 Summary

By implementing a relational database engine from scratch, we have
demystified how structured persistence systems operate beneath the
surface:

- **Relational Algebra** gives databases a rigorous mathematical
  foundation. Its property of closure allows queries to be composed,
  nested, and restructured cleanly.
- **The Iterator Model (Volcano)** processes data one tuple at a time
  across composable operator nodes, keeping memory overhead minimal.
- **Indexing** replaces $O(N)$ sequential scans with logarithmic
  $O(\log N)$ tree searches, turning full-table sweeps into
  lightning-fast lookups.
- **ACID Guarantees** ensure that our persistent data remains resilient,
  consistent, and correct, even when operations fail midway through
  execution.

### 16.14 Programming Challenges

#### Exercise 16.1: The Nested-Loop Equi-Join ($\bowtie$) Operator

**Context and Theory:**

The relational natural join and equi-join combine tuples from two
relations $R$ and $S$ on equality between specified attributes:

$$R \bowtie_{R.A = S.B} S = \sigma_{R.A = S.B}(R \times S)$$

In a Volcano-style iterator model, join operators are demand-driven via
`open()`, `next()`, and `close()`. The fundamental physical algorithm is
the **Nested-Loop Join**. For every outer tuple $r \in R$, the engine
iterates through all inner tuples $s \in S$. The schema of the resulting
relation has an arity equal to:

$$\text{arity}(R \bowtie S) = \text{arity}(R) + \text{arity}(S) - (\text{if natural join then } 1 \text{ else } 0)$$

The naive nested-loop join has a time complexity of $O(|R| \times |S|)$
and an auxiliary memory complexity of $O(1)$ beyond child iterator
state. The outer child is evaluated once, whereas the inner child is
reopened and rescanned $|R|$ times.

<figure id="fig:nested-loop">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0ODEuODk2MDMi
IGhlaWdodD0iNTkuODcyIiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDQ4MS44OTYwMyA1
OS44NzIiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJj
bGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0
cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsK
ICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2
Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFt
aWx5OiBJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTog
MTRweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDog
dHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsK
fQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVk
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5v
ZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cg
ewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJy
b3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRf
ZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRf
bWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2Jv
YiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7
Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgj
b3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1h
cmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmln
X29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5z
dmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJs
KCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0i
YXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9
IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+
CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJl
Zlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFy
dC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48
L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0i
MCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0i
NyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0i
NCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJr
ZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBt
YXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJz
ZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48
L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2
aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2Vy
SGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9
IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2Vy
PgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0i
NDgxLjg5NjAzIiBoZWlnaHQ9IjU5Ljg3MiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjEuNzQ2IiB5PSIyNC40NDQiID5bPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjEuNzQ2IiB5PSIzOC40MTIwMDMiID5bPC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjE1LjcxNDAwMSIgeT0iMjQuNDQ0IiA+Um93PC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE1LjcxNDAwMSIgeT0iMzguNDEyMDAzIiA+Um93
PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQzLjY1IiB5PSIyNC40NDQi
ID4wPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQzLjY1IiB5PSIzOC40
MTIwMDMiID4xPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjU3LjYxOCIg
eT0iMjQuNDQ0IiA+XTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1Ny42
MTgiIHk9IjM4LjQxMjAwMyIgPl08L3RleHQ+CiAgPGxpbmUgeDE9IjY5Ljg0MDAwNCIgeTE9IjIw
Ljk1MiIgeDI9IjE1My42NDgwMSIgeTI9IjIwLjk1MiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAg
PHBvbHlnb24gcG9pbnRzPSIxNTMuNjQ4MDEsMTcuNDYwMDAxIDE2MC42MzIsMjAuOTUyIDE1My42
NDgwMSwyNC40NDQiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8bGluZSB4MT0iNjkuODQw
MDA0IiB5MT0iMzQuOTIwMDAyIiB4Mj0iMTUzLjY0ODAxIiB5Mj0iMzQuOTIwMDAyIiBjbGFzcz0i
c29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjE1My42NDgwMSwzMS40MjgwMDEgMTYw
LjYzMiwzNC45MjAwMDIgMTUzLjY0ODAxLDM4LjQxMjAwMyIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5
Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE2OS4zNjIiIHk9IjI0LjQ0NCIg
Pls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTY5LjM2MiIgeT0iMzgu
NDEyMDAzIiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMDIuMDU4
IiB5PSIyNC40NDQiID5dPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMw
Mi4wNTgiIHk9IjM4LjQxMjAwMyIgPl08L3RleHQ+CiAgPGxpbmUgeDE9IjMxNC4yOCIgeTE9IjIw
Ljk1MiIgeDI9IjM3MC4xNTIiIHkyPSIyMC45NTIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxw
b2x5Z29uIHBvaW50cz0iMzcwLjE1MiwxNy40NjAwMDEgMzc3LjEzNjAyLDIwLjk1MiAzNzAuMTUy
LDI0LjQ0NCIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxsaW5lIHgxPSIzMTQuMjgiIHkx
PSIzNC45MjAwMDIiIHgyPSIzNzAuMTUyIiB5Mj0iMzQuOTIwMDAyIiBjbGFzcz0ic29saWQiPjwv
bGluZT4KICA8cG9seWdvbiBwb2ludHM9IjM3MC4xNTIsMzEuNDI4MDAxIDM3Ny4xMzYwMiwzNC45
MjAwMDIgMzcwLjE1MiwzOC40MTIwMDMiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxODMuMzMiIHk9IjM4LjQxMjAwMyIgPlJlc2V0PC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIyNS4yMzQwMSIgeT0iMzguNDEy
MDAzIiA+JmFtcDs8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjM5LjIw
MjAxIiB5PSIzOC40MTIwMDMiID5SZXNjYW48L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMjg4LjA5IiB5PSIzOC40MTIwMDMiID5TPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjEuNzQ2IiB5PSIxMC40NzYiID5PdXRlciBMb29wIChSKSAgICAgICAg
IElubmVyIExvb3AgKFMpICAgICAgICAgICAgICAgICBPdXRwdXQgTWF0Y2g8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTgzLjMzIiB5PSIyNC40NDQiID5TY2FuIFNfMCAu
LiBTX208L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzg1Ljg2NiIgeT0i
MjQuNDQ0IiA+WWllbGQgKHIg4oiYIHMpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjM4NS44NjYiIHk9IjM4LjQxMjAwMyIgPllpZWxkIChyIOKImCBzKTwvdGV4dD4KPC9z
dmc+Cg==
" alt="" />
<figcaption>Figure 6: Time Complexity of Nested Loop</figcaption>
</figure>

**Specification:**

Implement a `NestedLoopJoinNode` class inheriting from `ExecutionNode`.
The constructor must accept two child execution nodes, the outer join
column name, and the inner join column name. The output schema must
merge all outer columns with all inner columns, renaming or
deduplicating the join key as appropriate.

``` cpp
class NestedLoopJoinNode : public ExecutionNode {
public:
    NestedLoopJoinNode(std::unique_ptr<ExecutionNode> outer,
                       std::unique_ptr<ExecutionNode> inner,
                       std::string outer_col, std::string inner_col);
    void open() override;
    std::optional<Row> next() override;
    const Schema& output_schema() const override { return m_output_schema; }
private:
    std::unique_ptr<ExecutionNode> m_outer, m_inner;
    std::string m_outer_col_name, m_inner_col_name;
    size_t m_outer_col_idx{0}, m_inner_col_idx{0};
    std::optional<Row> m_current_outer_row;
    Schema m_output_schema;
};
```

**Verification Tasks:**

1.  **Disjoint Sets Match:** Populate relation $R$ with keys $\{1, 2\}$
    and relation $S$ with keys $\{3, 4\}$. Verify that `next()` returns
    `std::nullopt` on the first invocation ($0$ rows emitted).
2.  **One-to-Many Join:** Populate relation $R$ (Departments: `id`,
    `dept_name`) with `(1, "Systems")`, and $S$ (Engineers: `id`,
    `dept_id`, `name`) with `(101, 1, "Ada")` and `(102, 1, "Alan")`.
    Join on $R.\text{id} = S.\text{dept\_id}$. Verify that exactly two
    concatenated tuples are emitted: `[1, "Systems", 101, "Ada"]` and
    `[1, "Systems", 102, "Alan"]`.
3.  **Inner Rescan Verification:** Populate $R$ with $3$ rows and $S$
    with $4$ matching rows. Verify that `m_inner->open()` is invoked
    exactly $3$ times and that precisely $12$ joined tuples are emitted.

#### Exercise 16.2: Constant-Time Hash Indexing Engine

**Context and Theory:**

While balanced binary search trees (such as AVL or Red-Black trees)
guarantee $O(\log N)$ worst-case lookups and support efficient range
queries ($[k_{\text{low}}, k_{\text{high}}]$), point queries
(`WHERE id = 42`) can theoretically execute in amortised $O(1)$ time
using a **Hash Index**.

A hash index (see fig. 7) evaluates an attribute value through a hash
function $h(k): \text{Domain} \to \mathbb{N}$, mapping the key into an
array of collision buckets. Secondary indexing requires chaining,
because non-unique attributes map a single key to multiple matching
`RowID` instances. The primary operational disadvantage is the complete
loss of ordering: a hash index cannot accelerate range predicates (`<`,
`>`, `<=`, `>=`).

<figure id="fig:hash-lookup">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI0NzdwdCIgaGVpZ2h0PSI0NHB0Igogdmlld0JveD0iMC4wMCAwLjAw
IDQ3Ny4wMCA0NC4wMCIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIiB4bWxuczp4
bGluaz0iaHR0cDovL3d3dy53My5vcmcvMTk5OS94bGluayI+CjxnIGlkPSJncmFwaDAiIGNsYXNz
PSJncmFwaCIgdHJhbnNmb3JtPSJzY2FsZSgxIDEpIHJvdGF0ZSgwKSB0cmFuc2xhdGUoNCA0MCki
Pgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0ga2V5IC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNzPSJub2Rl
Ij4KPHRpdGxlPmtleTwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2si
IHBvaW50cz0iODQsLTM2IDAsLTM2IDAsMCA4NCwwIDg0LC0zNiIvPgo8dGV4dCB0ZXh0LWFuY2hv
cj0ibWlkZGxlIiB4PSI0MiIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBN
b25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+S2V5OiAmcXVvdDtTeXN0ZW1zJnF1b3Q7PC90
ZXh0Pgo8L2c+CjwhLS0gbW9kdWxvIC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2RlIj4KPHRp
dGxlPm1vZHVsbzwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBv
aW50cz0iMjA4LC0zNiAxMTMsLTM2IDExMywwIDIwOCwwIDIwOCwtMzYiLz4KPHRleHQgdGV4dC1h
bmNob3I9Im1pZGRsZSIgeD0iMTYwLjUiIHk9Ii0xNS41IiBmb250LWZhbWlseT0iSW9zZXZrYSBD
aGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPmgoS2V5KSAlIEJ1Y2tldHM8L3Rl
eHQ+CjwvZz4KPCEtLSBrZXkmIzQ1OyZndDttb2R1bG8gLS0+CjxnIGlkPSJlZGdlMSIgY2xhc3M9
ImVkZ2UiPgo8dGl0bGU+a2V5JiM0NTsmZ3Q7bW9kdWxvPC90aXRsZT4KPHBhdGggZmlsbD0ibm9u
ZSIgc3Ryb2tlPSJibGFjayIgZD0iTTg0LjA3LC0xOEM5MC4wMywtMTggOTYuMjYsLTE4IDEwMi40
NiwtMTgiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTAy
LjgzLC0yMS41IDExMi44MywtMTggMTAyLjgzLC0xNC41IDEwMi44MywtMjEuNSIvPgo8L2c+Cjwh
LS0gYnVja2V0IC0tPgo8ZyBpZD0ibm9kZTMiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmJ1Y2tldDwv
dGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMzAwLC0z
NiAyMzcsLTM2IDIzNywwIDMwMCwwIDMwMCwtMzYiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRs
ZSIgeD0iMjY4LjUiIHk9Ii0xNS41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBN
ZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPkJ1Y2tldCBbM108L3RleHQ+CjwvZz4KPCEtLSBtb2R1
bG8mIzQ1OyZndDtidWNrZXQgLS0+CjxnIGlkPSJlZGdlMiIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+
bW9kdWxvJiM0NTsmZ3Q7YnVja2V0PC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJi
bGFjayIgZD0iTTIwOC4zLC0xOEMyMTQuMzMsLTE4IDIyMC40NywtMTggMjI2LjQsLTE4Ii8+Cjxw
b2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjIyNi43OCwtMjEuNSAy
MzYuNzgsLTE4IDIyNi43OCwtMTQuNSAyMjYuNzgsLTIxLjUiLz4KPC9nPgo8IS0tIHJlc3VsdHMg
LS0+CjxnIGlkPSJub2RlNCIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+cmVzdWx0czwvdGl0bGU+Cjxw
b2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iNDY5LC0zNiAzMjksLTM2
IDMyOSwwIDQ2OSwwIDQ2OSwtMzYiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMzk5
IiB5PSItMTUuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250
LXNpemU9IjEwLjAwIj5bIFJvd0lEOiAxMDEsIFJvd0lEOiAxMDQgXTwvdGV4dD4KPC9nPgo8IS0t
IGJ1Y2tldCYjNDU7Jmd0O3Jlc3VsdHMgLS0+CjxnIGlkPSJlZGdlMyIgY2xhc3M9ImVkZ2UiPgo8
dGl0bGU+YnVja2V0JiM0NTsmZ3Q7cmVzdWx0czwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0
cm9rZT0iYmxhY2siIGQ9Ik0zMDAuMjgsLTE4QzMwNS45OCwtMTggMzEyLjE3LC0xOCAzMTguNTcs
LTE4Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjMxOC43
NywtMjEuNSAzMjguNzcsLTE4IDMxOC43NywtMTQuNSAzMTguNzcsLTIxLjUiLz4KPC9nPgo8L2c+
Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 7: Process of Hash Index Lookup</figcaption>
</figure>

**Specification:**

Create a `HashIndex` class that implements point lookups using
`std::unordered_map` with a custom hash functor capable of hashing
`CoreDB::Value`. Integrate `HashIndex` as an alternative indexing
backend in `Table`.

``` cpp
struct ValueHasher {
    std::size_t operator()(const Value& v) const noexcept {
        return std::visit([](const auto& val) -> std::size_t {
            using T = std::decay_t<decltype(val)>;
            if constexpr (std::is_same_v<T, std::monostate>) return 0;
            else if constexpr (std::is_same_v<T, std::string>) 
                return std::hash<std::string>{}(val);
            else return std::hash<double>{}(static_cast<double>(val));
        }, v.raw_data());
    }
};

class HashIndex {
public:
    void insert(const Value& key, RowID id);
    [[nodiscard]] std::vector<RowID> find_exact(const Value& key) const;
    void remove(const Value& key, RowID id);
private:
    std::unordered_map<Value, std::vector<RowID>, ValueHasher> m_buckets;
};
```

**Verification Tasks:**

1.  **Multi-Value Point Lookup:** Insert $10,000$ rows where $500$ rows
    share the department key `"Compilers"`. Verify that
    `find_exact("Compilers")` returns a vector of exactly $500$ distinct
    `RowID` references.
2.  **Key Absence Invariant:** Query a non-existent key `"Distributed"`.
    Verify that an empty `std::vector<RowID>` is returned in
    sub-microsecond time without generating exceptions or dangling
    bucket entries.
3.  **Collision Chain Integrity:** Insert two distinct string values
    engineered or selected to produce identical hash codes modulo bucket
    count. Verify that both distinct keys return their exact,
    non-overlapping `RowID` sets.

#### Exercise 16.3: Relational Mutation with Index Synchronisation (`DELETE` and `UPDATE`)

**Context and Theory:**

In a production database, mutating stored records requires strict
synchronisation between the primary row storage (the heap) and all
auxiliary search indices.

When a row is removed via `DELETE`:

1.  The row slot in the table heap must be deactivated or tombstoned
    ($O(1)$).
2.  For *every* index declared on the table, the old value must be
    deleted from the tree or hash structure ($O(K \log N)$ where $K$ is
    the number of indices).

When a row is modified via `UPDATE`:

If an updated column is indexed, the old key must be excised from the
index and the new key inserted. If a constraint (such as `PRIMARY KEY`
uniqueness) is violated during an `UPDATE`, the engine must abort and
revert all partially applied mutations.

**Specification:**

Extend `Table` to support `delete_records` and `update_records`. Both
functions must accept a `Predicate` and operate transactionally,
updating all registered BST and Hash indices while logging changes to
the undo log.

``` cpp
class Table {
    // ... existing definitions ...
public:
    size_t delete_records(const Predicate& predicate);
    size_t update_records(const Predicate& predicate, 
                          const std::string& column_name, 
                          const Value& new_value);
private:
    void remove_from_indices(RowID id, const Row& row);
    void insert_into_indices(RowID id, const Row& row);
};
```

**Verification Tasks:**

1.  **Index Eviction on Delete:** Insert $1,000$ records. Delete rows
    where `dept = 'Systems'`. Verify that scanning the table returns no
    rows matching the predicate, and that `index->find_exact("Systems")`
    returns an empty array.
2.  **Index Key Mutation on Update:** Update row with ID $103$ to set
    `dept = 'Cryptography'`. Verify that
    `index->find_exact("Cryptography")` yields row ID $103$, while a
    lookup for its previous department yields nothing.
3.  **Rollback on Constraint Violation:** Attempt an update that sets
    the primary key of an existing row to collide with another active
    row. Verify that the operation throws a constraint violation and
    that neither the table nor the indices retain the mutated key.

#### Exercise 16.4: Volcano Aggregate Pipeline (`SUM`, `COUNT`, `AVG`, `MIN`, `MAX`)

**Context and Theory:**

Relational algebra includes the extended generalised projection operator
$\mathcal{G}_{f_1(A_1), \dots, f_k(A_k)}(R)$, which computes aggregate
statistics over relations.

In contrast to selection ($\sigma$) and projection ($\pi$), aggregate
operators are **pipeline breakers**. A selection operator streams tuples
on-the-fly ($O(1)$ transient memory); an aggregate operator cannot yield
its first output tuple until it has consumed its child iterator to
exhaustion (`EOF`), as shown in fig. 8. The algorithm maintains an
accumulator state $(c, s, m_{\min}, m_{\max})$:

$$\text{Count: } c_{i} = c_{i-1} + 1 \qquad \text{Sum: } s_{i} = s_{i-1} + v_i \qquad \text{Avg: } \mu_i = \frac{s_i}{c_i}$$

<figure id="fig:aggregate-operators">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1NjUuNzA0MDQi
IGhlaWdodD0iNTkuODcyIiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDU2NS43MDQwNCA1
OS44NzIiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJj
bGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0
cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsK
ICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2
Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFt
aWx5OiBJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTog
MTRweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDog
dHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsK
fQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVk
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5v
ZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cg
ewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJy
b3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRf
ZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRf
bWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2Jv
YiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7
Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgj
b3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1h
cmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmln
X29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5z
dmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJs
KCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0i
YXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9
IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+
CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJl
Zlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFy
dC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48
L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0i
MCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0i
NyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0i
NCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJr
ZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBt
YXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJz
ZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48
L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2
aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2Vy
SGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9
IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2Vy
PgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0i
NTY1LjcwNDA0IiBoZWlnaHQ9IjU5Ljg3MiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+Q2hpbGQ8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTAuNjM0MDAzIiB5PSIxMC40NzYiID5TdHJlYW06PC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEwNi41MDYwMDQiIHk9IjEwLjQ3NiIg
Pls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTIwLjQ3NDAxIiB5PSIx
MC40NzYiID5Sb3c8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTQ4LjQx
IiB5PSIxMC40NzYiID5dPC90ZXh0PgogIDxsaW5lIHgxPSIxNjAuNjMyIiB5MT0iNi45ODQiIHgy
PSIxNzQuNiIgeTI9IjYuOTg0IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2lu
dHM9IjE3NC42LDMuNDkyIDE4MS41ODQsNi45ODQgMTc0LjYsMTAuNDc2IiBjbGFzcz0iZmlsbGVk
Ij48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTkwLjMxNDAxIiB5
PSIxMC40NzYiID5bPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIwNC4y
ODIwMSIgeT0iMTAuNDc2IiA+Um93PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjIzMi4yMTgiIHk9IjEwLjQ3NiIgPl08L3RleHQ+CiAgPGxpbmUgeDE9IjI0NC40NCIgeTE9
IjYuOTg0IiB4Mj0iMjU4LjQwODAyIiB5Mj0iNi45ODQiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgog
IDxwb2x5Z29uIHBvaW50cz0iMjU4LjQwODAyLDMuNDkyIDI2NS4zOTIsNi45ODQgMjU4LjQwODAy
LDEwLjQ3NiIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjI3NC4xMjIiIHk9IjEwLjQ3NiIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iMjg4LjA5IiB5PSIxMC40NzYiID5Sb3c8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzE2LjAyNiIgeT0iMTAuNDc2IiA+XTwvdGV4dD4KICA8bGlu
ZSB4MT0iMzI4LjI0ODAyIiB5MT0iNi45ODQiIHgyPSIzNDIuMjE2IiB5Mj0iNi45ODQiIGNsYXNz
PSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMzQyLjIxNiwzLjQ5MiAzNDkuMiw2
Ljk4NCAzNDIuMjE2LDEwLjQ3NiIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxwb2x5Z29u
IHBvaW50cz0iNDA1LjA3MjAyLDMxLjQyODAwMSA0MTIuMDU2LDM0LjkyMDAwMiA0MDUuMDcyMDIs
MzguNDEyMDAzIiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iNDIwLjc4NiIgeT0iMzguNDEyMDAzIiA+WWllbGQ8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDYyLjY5IiB5PSIzOC40MTIwMDMiID5BZ2dyZWdhdGVk
PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjUzOS41MTQwNCIgeT0iMzgu
NDEyMDAzIiA+Um93PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM1Ny45
MzAwMiIgeT0iMTAuNDc2IiA+RU9GPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjguNzMwMDAwNSIgeT0iMzguNDEyMDAzIiA+QWNjdW11bGF0b3I6IChjb3VudD0zLCBzdW09
MjQwLjAsIGF2Zz04MC4wKTwvdGV4dD4KICA8Zz4KICAgIDxsaW5lIHgxPSIzNjYuNjYiIHkxPSIx
My45NjgiIHgyPSIzNjYuNjYiIHkyPSIzNC45MjAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgog
ICAgPGxpbmUgeDE9IjMyMS4yNjQiIHkxPSIzNC45MjAwMDIiIHgyPSI0MDUuMDcyMDIiIHkyPSIz
NC45MjAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 8: Operation of Aggregate Pipeline</figcaption>
</figure>

**Specification:**

Implement an `AggregateNode` conforming to the `ExecutionNode`
interface. The constructor must take a child execution node, an
aggregation function enumeration, and the target column name. When
evaluated, it must emit exactly one row containing the computed
aggregate scalar.

``` cpp
enum class AggregateType { Count, Sum, Avg, Min, Max };

class AggregateNode : public ExecutionNode {
public:
    AggregateNode(std::unique_ptr<ExecutionNode> child, 
                  AggregateType agg_type, 
                  std::string target_col);
    void open() override;
    std::optional<Row> next() override;
    const Schema& output_schema() const override { return m_schema; }
private:
    std::unique_ptr<ExecutionNode> m_child;
    AggregateType m_agg_type;
    std::string m_target_col;
    Schema m_schema;
    bool m_has_emitted{false};
};
```

**Verification Tasks:**

1.  **Zero-Row Aggregation Invariant:** Pass an empty child iterator
    (filtered by an unsatisfiable predicate) to an `AggregateNode`
    configured for `COUNT`. Verify that it emits a single row containing
    `Value(int64_t(0))`.
2.  **Floating-Point Precision:** Feed $10,000$ values drawn uniformly
    from $[0.0, 100.0]$ with an analytical mean $\mu = 50.0$. Verify
    that `SUM`, `AVG`, `MIN`, and `MAX` match calculations within an
    absolute floating-point tolerance $\epsilon = 10^{-5}$.
3.  **Composition:** Chain `PhysicalScanNode` $\to$ `ProjectionNode`
    $\to$ `AggregateNode`. Verify that
    `SELECT AVG(salary) FROM engineers WHERE dept = 'Compilers'` returns
    the expected scalar.

#### Exercise 16.5: Recursive Boolean Expression Trees (DNF and CNF)

**Context and Theory:**

Basic database engines limit selection to a single column predicate
($A \text{ op } c$). Production engines evaluate compound propositional
formulas composed of conjunctions ($\land$), disjunctions ($\lor$), and
negations ($\neg$).

Under Codd’s relational model, queries operate under **Three-Valued
Logic (3VL)** when `NULL` values are present, evaluating propositions to
`TRUE`, `FALSE`, or `UNKNOWN`. For selection filters, tuples are
retained if and only if the expression evaluates to `TRUE`. A diagram of
a Boolean Expression Tree suitable to be used with 3VL is shown in
fig. 9.

<figure id="fig:three-value-logic">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI0MTFwdCIgaGVpZ2h0PSIyMzlwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCA0MTAuODcgMjM5LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDIz
NSkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gb3IgLS0+CjxnIGlkPSJub2RlMSIgY2xhc3M9Im5v
ZGUiPgo8dGl0bGU+b3I8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNr
IiBjeD0iMjM2LjA5IiBjeT0iLTIxMyIgcng9IjM0LjI5IiByeT0iMTgiLz4KPHRleHQgdGV4dC1h
bmNob3I9Im1pZGRsZSIgeD0iMjM2LjA5IiB5PSItMjEwLjUiIGZvbnQtZmFtaWx5PSJJb3Nldmth
IENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+T1IgTm9kZTwvdGV4dD4KPC9n
Pgo8IS0tIGFuZCAtLT4KPGcgaWQ9Im5vZGUyIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5hbmQ8L3Rp
dGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0iMTg4LjA5IiBjeT0i
LTE0OCIgcng9IjM4LjciIHJ5PSIxOCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIx
ODguMDkiIHk9Ii0xNDUuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVt
IiBmb250LXNpemU9IjEwLjAwIj5BTkQgTm9kZTwvdGV4dD4KPC9nPgo8IS0tIG9yJiM0NTsmZ3Q7
YW5kIC0tPgo8ZyBpZD0iZWRnZTEiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPm9yJiM0NTsmZ3Q7YW5k
PC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTIyMy45NywtMTk2
LjFDMjE4LjY3LC0xODkuMTQgMjEyLjMzLC0xODAuODIgMjA2LjQ3LC0xNzMuMTMiLz4KPHBvbHln
b24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMjA5LjIxLC0xNzAuOTQgMjAw
LjM2LC0xNjUuMTEgMjAzLjY0LC0xNzUuMTkgMjA5LjIxLC0xNzAuOTQiLz4KPC9nPgo8IS0tIGVx
dWFsMSAtLT4KPGcgaWQ9Im5vZGUzIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5lcXVhbDE8L3RpdGxl
Pgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0iMjg1LjA5IiBjeT0iLTE0
OCIgcng9IjQwLjc0IiByeT0iMTgiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjg1
LjA5IiB5PSItMTQ1LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIg
Zm9udC1zaXplPSIxMC4wMCI+RXF1YWwgTm9kZTwvdGV4dD4KPC9nPgo8IS0tIG9yJiM0NTsmZ3Q7
ZXF1YWwxIC0tPgo8ZyBpZD0iZWRnZTIiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPm9yJiM0NTsmZ3Q7
ZXF1YWwxPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTI0OC40
NSwtMTk2LjFDMjUzLjc4LC0xODkuMjUgMjYwLjE0LC0xODEuMDggMjY2LjA0LC0xNzMuNDgiLz4K
PHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMjY4LjgzLC0xNzUu
NTkgMjcyLjIxLC0xNjUuNTUgMjYzLjMxLC0xNzEuMyAyNjguODMsLTE3NS41OSIvPgo8L2c+Cjwh
LS0gZ3JlYXRlciAtLT4KPGcgaWQ9Im5vZGU0IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5ncmVhdGVy
PC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9IjExOC4wOSIg
Y3k9Ii04MyIgcng9IjI5LjM3IiByeT0iMTgiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIg
eD0iMTE4LjA5IiB5PSItODAuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVk
aXVtIiBmb250LXNpemU9IjEwLjAwIj4mZ3Q7IE5vZGU8L3RleHQ+CjwvZz4KPCEtLSBhbmQmIzQ1
OyZndDtncmVhdGVyIC0tPgo8ZyBpZD0iZWRnZTMiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmFuZCYj
NDU7Jmd0O2dyZWF0ZXI8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBk
PSJNMTcxLjE0LC0xMzEuNzVDMTYyLjIsLTEyMy43IDE1MS4xLC0xMTMuNzEgMTQxLjM2LC0xMDQu
OTUiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTQzLjY5
LC0xMDIuMzMgMTMzLjkyLC05OC4yNSAxMzkuMDEsLTEwNy41NCAxNDMuNjksLTEwMi4zMyIvPgo8
L2c+CjwhLS0gZXF1YWwyIC0tPgo8ZyBpZD0ibm9kZTUiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmVx
dWFsMjwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIxOTIu
MDkiIGN5PSItODMiIHJ4PSIyNyIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUi
IHg9IjE5Mi4wOSIgeT0iLTgwLjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1l
ZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+PT08L3RleHQ+CjwvZz4KPCEtLSBhbmQmIzQ1OyZndDtl
cXVhbDIgLS0+CjxnIGlkPSJlZGdlNCIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+YW5kJiM0NTsmZ3Q7
ZXF1YWwyPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTE4OS4x
OCwtMTI5Ljc4QzE4OS41NCwtMTI0LjA5IDE4OS45NSwtMTE3LjYxIDE5MC4zNSwtMTExLjM0Ii8+
Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjE5My44NSwtMTEx
LjQgMTkwLjk5LC0xMDEuMTkgMTg2Ljg3LC0xMTAuOTUgMTkzLjg1LC0xMTEuNCIvPgo8L2c+Cjwh
LS0gZGVwdCAtLT4KPGcgaWQ9Im5vZGUxMCIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+ZGVwdDwvdGl0
bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIyODIuMDkiIGN5PSIt
ODMiIHJ4PSIzOC43IiByeT0iMTgiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjgy
LjA5IiB5PSItODAuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBm
b250LXNpemU9IjEwLjAwIj5Db2woJnF1b3Q7ZGVwdCZxdW90Oyk8L3RleHQ+CjwvZz4KPCEtLSBl
cXVhbDEmIzQ1OyZndDtkZXB0IC0tPgo8ZyBpZD0iZWRnZTkiIGNsYXNzPSJlZGdlIj4KPHRpdGxl
PmVxdWFsMSYjNDU7Jmd0O2RlcHQ8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJs
YWNrIiBkPSJNMjg0LjI3LC0xMjkuNzhDMjgzLjk5LC0xMjQuMDkgMjgzLjY5LC0xMTcuNjEgMjgz
LjM5LC0xMTEuMzQiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50
cz0iMjg2Ljg4LC0xMTEuMDIgMjgyLjksLTEwMS4xOSAyNzkuODgsLTExMS4zNSAyODYuODgsLTEx
MS4wMiIvPgo8L2c+CjwhLS0gc2VjdXJpdHkgLS0+CjxnIGlkPSJub2RlMTEiIGNsYXNzPSJub2Rl
Ij4KPHRpdGxlPnNlY3VyaXR5PC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJi
bGFjayIgY3g9IjM3MS4wOSIgY3k9Ii04MyIgcng9IjMxLjU4IiByeT0iMTgiLz4KPHRleHQgdGV4
dC1hbmNob3I9Im1pZGRsZSIgeD0iMzcxLjA5IiB5PSItODAuNSIgZm9udC1mYW1pbHk9Iklvc2V2
a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5TZWN1cml0eTwvdGV4dD4K
PC9nPgo8IS0tIGVxdWFsMSYjNDU7Jmd0O3NlY3VyaXR5IC0tPgo8ZyBpZD0iZWRnZTEwIiBjbGFz
cz0iZWRnZSI+Cjx0aXRsZT5lcXVhbDEmIzQ1OyZndDtzZWN1cml0eTwvdGl0bGU+CjxwYXRoIGZp
bGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0zMDUuNDcsLTEzMi4wN0MzMTcuMDEsLTEyMy42
MSAzMzEuNTksLTExMi45MyAzNDQuMDgsLTEwMy43OSIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIg
c3Ryb2tlPSJibGFjayIgcG9pbnRzPSIzNDYuMzIsLTEwNi40OCAzNTIuMzIsLTk3Ljc1IDM0Mi4x
OSwtMTAwLjgzIDM0Ni4zMiwtMTA2LjQ4Ii8+CjwvZz4KPCEtLSBhZ2UgLS0+CjxnIGlkPSJub2Rl
NiIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+YWdlPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIg
c3Ryb2tlPSJibGFjayIgY3g9IjM3LjA5IiBjeT0iLTE4IiByeD0iMzcuMTciIHJ5PSIxOCIvPgo8
dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIzNy4wOSIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5
PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+Q29sICgmcXVv
dDthZ2UmcXVvdDspPC90ZXh0Pgo8L2c+CjwhLS0gZ3JlYXRlciYjNDU7Jmd0O2FnZSAtLT4KPGcg
aWQ9ImVkZ2U1IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5ncmVhdGVyJiM0NTsmZ3Q7YWdlPC90aXRs
ZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTEwMC41MSwtNjguMzNDODku
ODUsLTYwLjA0IDc2LjAzLC00OS4yOSA2NC4wMSwtMzkuOTQiLz4KPHBvbHlnb24gZmlsbD0iYmxh
Y2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iNjYuMDgsLTM3LjEyIDU2LjA0LC0zMy43NCA2MS43
OCwtNDIuNjQgNjYuMDgsLTM3LjEyIi8+CjwvZz4KPCEtLSAyNSAtLT4KPGcgaWQ9Im5vZGU3IiBj
bGFzcz0ibm9kZSI+Cjx0aXRsZT4yNTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9r
ZT0iYmxhY2siIGN4PSIxMTkuMDkiIGN5PSItMTgiIHJ4PSIyNyIgcnk9IjE4Ii8+Cjx0ZXh0IHRl
eHQtYW5jaG9yPSJtaWRkbGUiIHg9IjExOS4wOSIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJJb3Nl
dmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+MjU8L3RleHQ+CjwvZz4K
PCEtLSBncmVhdGVyJiM0NTsmZ3Q7MjUgLS0+CjxnIGlkPSJlZGdlNiIgY2xhc3M9ImVkZ2UiPgo8
dGl0bGU+Z3JlYXRlciYjNDU7Jmd0OzI1PC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tl
PSJibGFjayIgZD0iTTExOC4zNiwtNjQuNzhDMTE4LjQ1LC01OS4wOSAxMTguNTUsLTUyLjYxIDEx
OC42NSwtNDYuMzQiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50
cz0iMTIyLjE1LC00Ni4yNSAxMTguODEsLTM2LjE5IDExNS4xNSwtNDYuMTQgMTIyLjE1LC00Ni4y
NSIvPgo8L2c+CjwhLS0gY29sIC0tPgo8ZyBpZD0ibm9kZTgiIGNsYXNzPSJub2RlIj4KPHRpdGxl
PmNvbDwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIxOTIu
MDkiIGN5PSItMTgiIHJ4PSIyNyIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUi
IHg9IjE5Mi4wOSIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1l
ZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+Q29sLi4uPC90ZXh0Pgo8L2c+CjwhLS0gZXF1YWwyJiM0
NTsmZ3Q7Y29sIC0tPgo8ZyBpZD0iZWRnZTciIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmVxdWFsMiYj
NDU7Jmd0O2NvbDwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0x
OTIuMDksLTY0Ljc4QzE5Mi4wOSwtNTkuMDkgMTkyLjA5LC01Mi42MSAxOTIuMDksLTQ2LjM0Ii8+
Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjE5NS41OSwtNDYu
MTkgMTkyLjA5LC0zNi4xOSAxODguNTksLTQ2LjE5IDE5NS41OSwtNDYuMTkiLz4KPC9nPgo8IS0t
IHN5c3RlbXMgLS0+CjxnIGlkPSJub2RlOSIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+c3lzdGVtczwv
dGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIyNzMuMDkiIGN5
PSItMTgiIHJ4PSIzNi40OSIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9
IjI3My4wOSIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1
bSIgZm9udC1zaXplPSIxMC4wMCI+JnF1b3Q7U3lzdGVtcyZxdW90OzwvdGV4dD4KPC9nPgo8IS0t
IGVxdWFsMiYjNDU7Jmd0O3N5c3RlbXMgLS0+CjxnIGlkPSJlZGdlOCIgY2xhc3M9ImVkZ2UiPgo8
dGl0bGU+ZXF1YWwyJiM0NTsmZ3Q7c3lzdGVtczwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0
cm9rZT0iYmxhY2siIGQ9Ik0yMDkuMjYsLTY4LjY0QzIxOS45OCwtNjAuMyAyMzQsLTQ5LjQgMjQ2
LjE3LC0zOS45NCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRz
PSIyNDguNDksLTQyLjU2IDI1NC4yNCwtMzMuNjYgMjQ0LjE5LC0zNy4wNCAyNDguNDksLTQyLjU2
Ii8+CjwvZz4KPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 9: Boolean Expression Tree</figcaption>
</figure>

**Specification:**

Design an Abstract Syntax Tree (AST) hierarchy for boolean expressions
based on a `BooleanExpr` base class. Implement terminal nodes
(`ComparisonPredicate`) and non-terminal combinators (`AndExpr`,
`OrExpr`, `NotExpr`). Integrate this tree into `PhysicalScanNode`.

``` cpp
class BooleanExpr {
public:
    virtual ~BooleanExpr() = default;
    [[nodiscard]] virtual bool evaluate(const Row& row, const Schema& schema) const = 0;
};

class BinaryLogicExpr : public BooleanExpr {
public:
    enum class LogicOp { And, Or };
    BinaryLogicExpr(std::unique_ptr<BooleanExpr> left, LogicOp op, std::unique_ptr<BooleanExpr> right);
    bool evaluate(const Row& row, const Schema& schema) const override;
private:
    std::unique_ptr<BooleanExpr> m_left, m_right;
    LogicOp m_op;
};
```

**Verification Tasks:**

1.  **De Morgan’s Laws Equivalence:** Construct two filters over a
    table: $P_1 = \neg (A \land B)$ and $P_2 = (\neg A) \lor (\neg B)$.
    Verify across $1,000$ diverse rows that $P_1$ and $P_2$ emit
    identical sets of `RowID` instances.
2.  **Short-Circuit Evaluation:** Construct an expression
    $(A = 1) \lor (B / 0 = 1)$. Configure the left branch to evaluate to
    `TRUE`. Verify that the right branch is never evaluated, avoiding a
    division-by-zero exception.
3.  **Three-Valued Logic Null Check:** Evaluate `(NULL = NULL)` and
    `(age > 20 AND NULL)`. Verify that both evaluate to `UNKNOWN`
    (treated as `FALSE` by the selection filter) and that no rows are
    returned.

#### Exercise 16.6: Paged B+ Tree Leaf Traversal

**Context and Theory:**

Binary search trees (such as the AVL tree used in Section 16.8) suffer
from high tree heights ($h = \log_2 N$) and poor cache locality.
Production database storage engines rely on **B+ Trees**.

A B+ Tree differs from a standard B-Tree in two critical structural
ways:

1.  Internal nodes store only routing keys and child pointers; they
    never store row references.
2.  Leaf nodes store all key-payload pairs and are linked contiguously
    via `next` and `prev` pointers into a doubly-linked list.

This architecture enables $O(\log_B N)$ point lookups, followed by
$O(K)$ sequential memory traversals for range queries
$[k_{\text{start}}, k_{\text{end}}]$, avoiding parent node backtracking.

<figure id="fig:b-plus-tree">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSIzOTguMDg4IiBo
ZWlnaHQ9IjU5Ljg3MiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCAzOTguMDg4IDU5Ljg3
MiI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxNHB4
Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFu
c3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgou
c3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewog
IGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxs
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5l
bmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVu
X2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Bl
bl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Jp
Z19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJv
dyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5
Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0i
MiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJl
dmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9s
eWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAg
OCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBv
cmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiBy
PSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBp
ZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtl
cldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4K
ICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2ly
Y2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdC
b3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWln
aHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIg
Y3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAg
PC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSIzOTgu
MDg4IiBoZWlnaHQ9IjU5Ljg3MiI+PC9yZWN0PgogIDxwb2x5Z29uIHBvaW50cz0iMTc0LjYsMy40
OTIgMTY3LjYxNiw2Ljk4NCAxNzQuNiwxMC40NzYiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4K
ICA8bGluZSB4MT0iMTc0LjYiIHkxPSI2Ljk4NCIgeDI9IjIzMC40NzIiIHkyPSI2Ljk4NCIgY2xh
c3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIyMzAuNDcyLDMuNDkyIDIzNy40
NTYwMSw2Ljk4NCAyMzAuNDcyLDEwLjQ3NiIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxs
aW5lIHgxPSI2Ni4zNDgiIHkxPSIxMy45NjgiIHgyPSI2Ni4zNDgiIHkyPSIyNy45MzYiIGNsYXNz
PSJzb2xpZCI+PC9saW5lPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjUwLjYzNDAw
MyIgeT0iMzguNDEyMDAzIiA+Um93MDwvdGV4dD4KICA8bGluZSB4MT0iMTAxLjI2ODAwNSIgeTE9
IjEzLjk2OCIgeDI9IjEwMS4yNjgwMDUiIHkyPSIyNy45MzYiIGNsYXNzPSJzb2xpZCI+PC9saW5l
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijg1LjU1NCIgeT0iMzguNDEyMDAzIiA+
Um93MTwvdGV4dD4KICA8bGluZSB4MT0iMTM2LjE4OCIgeTE9IjEzLjk2OCIgeDI9IjEzNi4xODgi
IHkyPSIyNy45MzYiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjEyMC40NzQwMSIgeT0iMzguNDEyMDAzIiA+Um93MjwvdGV4dD4KICA8bGluZSB4
MT0iMzAzLjgwNDAyIiB5MT0iMTMuOTY4IiB4Mj0iMzAzLjgwNDAyIiB5Mj0iMjcuOTM2IiBjbGFz
cz0ic29saWQiPjwvbGluZT4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyODguMDki
IHk9IjM4LjQxMjAwMyIgPlJvdzM8L3RleHQ+CiAgPGxpbmUgeDE9IjMzOC43MjQiIHkxPSIxMy45
NjgiIHgyPSIzMzguNzI0IiB5Mj0iMjcuOTM2IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMjMuMDEiIHk9IjM4LjQxMjAwMyIgPlJvdzQ8L3Rl
eHQ+CiAgPGxpbmUgeDE9IjM3My42NDQiIHkxPSIxMy45NjgiIHgyPSIzNzMuNjQ0IiB5Mj0iMjcu
OTM2IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSIzNTcuOTMwMDIiIHk9IjM4LjQxMjAwMyIgPlJvdzU8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iOC43MzAwMDA1IiB5PSIxMC40NzYiID5MZWFmIFsgMTAgfCAyMCB8IDMw
IF08L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjQ2LjE4NiIgeT0iMTAu
NDc2IiA+TGVhZiBbIDQwIHwgNTAgfCA2MCBdPC90ZXh0Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 10: B+ Tree Layout</figcaption>
</figure>

**Specification:**

Implement an in-memory B+ Tree leaf node structure, `BPlusTreeLeafNode`,
with fixed capacity $B$. Support key insertion with node splitting, key
lookups, and range scanning using linked leaf sibling pointers.

``` cpp
template <typename Key, typename Value, size_t Capacity = 4>
struct BPlusTreeLeafNode {
    size_t size{0};
    std::array<Key, Capacity> keys;
    std::array<std::vector<Value>, Capacity> payloads;
    BPlusTreeLeafNode* next{nullptr};
    BPlusTreeLeafNode* prev{nullptr};

    [[nodiscard]] bool is_full() const noexcept { return size == Capacity; }
    size_t find_key_index(const Key& key) const;
    void insert_non_full(const Key& key, const Value& val);
    std::unique_ptr<BPlusTreeLeafNode> split();
};
```

**Verification Tasks:**

1.  **Node Split and Link Invariant:** Insert $5$ sorted keys into a
    leaf node with capacity $B = 4$. Verify that splitting creates a new
    leaf, distributes keys evenly, and sets `left->next == right` and
    `right->prev == left`.
2.  **Range Query via Sibling Links:** Populate a multi-leaf structure
    with integers $1 \dots 100$. Execute a range search for
    $25 \le k \le 75$. Verify that the scan finds key $25$ via index
    lookup and traverses sibling pointers directly to key $75$ without
    referencing parent nodes.
3.  **Reverse Range Traversal:** Utilise `prev` pointers to scan
    backwards from key $50$ down to $10$. Verify exact descending
    sequence retrieval.

#### Exercise 16.7: Cost-Based Optimiser and Selectivity Estimation

**Context and Theory:**

A query optimiser translates a logical relational algebra expression
into an efficient physical execution plan. When evaluating a selection
predicate $\sigma_{A = v}(R)$, the optimiser must decide between:

1.  **Full Sequential Scan:** Cost
    $C_{\text{seq}} = |R| \cdot c_{\text{cpu}}$
2.  **Index Scan:** Cost
    $C_{\text{idx}} = \log_2 N \cdot c_{\text{tree}} + (\text{Selectivity} \cdot |R|) \cdot (c_{\text{fetch}} + c_{\text{cpu}})$

**Selectivity ($S \in [0.0, 1.0]$)** represents the fraction of tuples
expected to satisfy the predicate. Assuming a uniform distribution over
domain cardinality $V(A, R)$:

$$S(A = v) = \frac{1}{|V(A, R)|} \qquad S(A > v) = \frac{\max(A) - v}{\max(A) - \min(A)}$$

If selectivity is high (e.g., $S > 0.3$), sequential scanning is faster
than index lookups due to cache prefetching and eliminating secondary
index pointer indirection, see also fig. 11.

<figure id="fig:selectivity-estimator">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIyMjZwdCIgaGVpZ2h0PSIyMThwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAyMjYuMDcgMjE4LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDIx
NCkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gcXVlcnkgLS0+CjxnIGlkPSJub2RlMSIgY2xhc3M9
Im5vZGUiPgo8dGl0bGU+cXVlcnk8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9
ImJsYWNrIiBwb2ludHM9IjE1NC42MywtMjEwIDYyLjYzLC0yMTAgNjIuNjMsLTE3NCAxNTQuNjMs
LTE3NCAxNTQuNjMsLTIxMCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxMDguNjMi
IHk9Ii0xODkuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250
LXNpemU9IjEwLjAwIj5RdWVyeTogU0VMRUNULi4uPC90ZXh0Pgo8L2c+CjwhLS0gZXN0aW1hdG9y
IC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmVzdGltYXRvcjwvdGl0bGU+
CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIxMDguNjMiIGN5PSItMTM0
IiByeD0iNjQuNTEiIHJ5PSIxOCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxMDgu
NjMiIHk9Ii0xMzEuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBm
b250LXNpemU9IjEwLjAwIj5TZWxlY3Rpdml0eSBFc3RpbWF0b3I8L3RleHQ+CjwvZz4KPCEtLSBx
dWVyeSYjNDU7Jmd0O2VzdGltYXRvciAtLT4KPGcgaWQ9ImVkZ2UxIiBjbGFzcz0iZWRnZSI+Cjx0
aXRsZT5xdWVyeSYjNDU7Jmd0O2VzdGltYXRvcjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0
cm9rZT0iYmxhY2siIGQ9Ik0xMDguNjMsLTE3My44OUMxMDguNjMsLTE3MC4yIDEwOC42MywtMTY2
LjIxIDEwOC42MywtMTYyLjI0Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNr
IiBwb2ludHM9IjExMi4xMywtMTYyIDEwOC42MywtMTUyIDEwNS4xMywtMTYyIDExMi4xMywtMTYy
Ii8+CjwvZz4KPCEtLSB0aHJlc2hvbGQxIC0tPgo8ZyBpZD0ibm9kZTMiIGNsYXNzPSJub2RlIj4K
PHRpdGxlPnRocmVzaG9sZDE8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJs
YWNrIiBjeD0iNTEuNjMiIGN5PSItNzYiIHJ4PSI0NS44MyIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQt
YW5jaG9yPSJtaWRkbGUiIHg9IjUxLjYzIiB5PSItNzMuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2Eg
Q2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5TICZsdDsgVGhyZXNob2xkPC90
ZXh0Pgo8L2c+CjwhLS0gZXN0aW1hdG9yJiM0NTsmZ3Q7dGhyZXNob2xkMSAtLT4KPGcgaWQ9ImVk
Z2UyIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5lc3RpbWF0b3ImIzQ1OyZndDt0aHJlc2hvbGQxPC90
aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTkxLjgzLC0xMTYuNUM4
Ni42OCwtMTExLjQ0IDgwLjkyLC0xMDUuNzggNzUuNCwtMTAwLjM2Ii8+Cjxwb2x5Z29uIGZpbGw9
ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9Ijc3LjY5LC05Ny43IDY4LjEsLTkzLjE5IDcy
Ljc4LC0xMDIuNjkgNzcuNjksLTk3LjciLz4KPC9nPgo8IS0tIHRocmVzaG9sZDIgLS0+CjxnIGlk
PSJub2RlNCIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+dGhyZXNob2xkMjwvdGl0bGU+CjxlbGxpcHNl
IGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIxNjYuNjMiIGN5PSItNzYiIHJ4PSI0OS40
IiByeT0iMTgiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTY2LjYzIiB5PSItNzMu
NSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEw
LjAwIj5TICZndDs9IFRocmVzaG9sZDwvdGV4dD4KPC9nPgo8IS0tIGVzdGltYXRvciYjNDU7Jmd0
O3RocmVzaG9sZDIgLS0+CjxnIGlkPSJlZGdlMyIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+ZXN0aW1h
dG9yJiM0NTsmZ3Q7dGhyZXNob2xkMjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIGQ9Ik0xMjUuNzEsLTExNi41QzEzMC45NSwtMTExLjQ0IDEzNi44MiwtMTA1Ljc4IDE0
Mi40MywtMTAwLjM2Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2lu
dHM9IjE0NS4xLC0xMDIuNjUgMTQ5Ljg2LC05My4xOSAxNDAuMjMsLTk3LjYyIDE0NS4xLC0xMDIu
NjUiLz4KPC9nPgo8IS0tIHNjYW4xIC0tPgo8ZyBpZD0ibm9kZTUiIGNsYXNzPSJub2RlIj4KPHRp
dGxlPnNjYW4xPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9
IjUwLjYzIiBjeT0iLTE4IiByeD0iNTAuNzUiIHJ5PSIxOCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0i
bWlkZGxlIiB4PSI1MC42MyIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBN
b25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+VXNlIEluZGV4IFNjYW48L3RleHQ+CjwvZz4K
PCEtLSB0aHJlc2hvbGQxJiM0NTsmZ3Q7c2NhbjEgLS0+CjxnIGlkPSJlZGdlNCIgY2xhc3M9ImVk
Z2UiPgo8dGl0bGU+dGhyZXNob2xkMSYjNDU7Jmd0O3NjYW4xPC90aXRsZT4KPHBhdGggZmlsbD0i
bm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTUxLjMyLC01Ny44OUM1MS4yNSwtNTQuMiA1MS4xOCwt
NTAuMjEgNTEuMTEsLTQ2LjI0Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNr
IiBwb2ludHM9IjU0LjYxLC00NS45NCA1MC45MywtMzYgNDcuNjEsLTQ2LjA2IDU0LjYxLC00NS45
NCIvPgo8L2c+CjwhLS0gc2NhbjIgLS0+CjxnIGlkPSJub2RlNiIgY2xhc3M9Im5vZGUiPgo8dGl0
bGU+c2NhbjI8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0i
MTY4LjYzIiBjeT0iLTE4IiByeD0iNDkuNCIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJt
aWRkbGUiIHg9IjE2OC42MyIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBN
b25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+VXNlIFRhYmxlIFNjYW48L3RleHQ+CjwvZz4K
PCEtLSB0aHJlc2hvbGQyJiM0NTsmZ3Q7c2NhbjIgLS0+CjxnIGlkPSJlZGdlNSIgY2xhc3M9ImVk
Z2UiPgo8dGl0bGU+dGhyZXNob2xkMiYjNDU7Jmd0O3NjYW4yPC90aXRsZT4KPHBhdGggZmlsbD0i
bm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTE2Ny4yNCwtNTcuODlDMTY3LjM3LC01NC4yIDE2Ny41
MSwtNTAuMjEgMTY3LjY1LC00Ni4yNCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJi
bGFjayIgcG9pbnRzPSIxNzEuMTYsLTQ2LjEyIDE2OC4wMiwtMzYgMTY0LjE2LC00NS44NyAxNzEu
MTYsLTQ2LjEyIi8+CjwvZz4KPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 11: Selectivity Estimator Schematic</figcaption>
</figure>

**Specification:**

Create a `CostBasedOptimizer` class that inspects `Table` catalog
statistics (row count, min/max values, distinct value counts). Given a
query predicate, estimate its selectivity, compute operational costs,
and select between `IndexScan` and `SequentialScan`.

``` cpp
struct TableStatistics {
    size_t total_rows{0};
    std::unordered_map<std::string, size_t> distinct_counts;
    std::unordered_map<std::string, std::pair<double, double>> numeric_bounds;
};

class CostBasedOptimizer {
public:
    explicit CostBasedOptimizer(TableStatistics stats);
    double estimate_selectivity(const Predicate& pred) const;
    bool should_use_index(const Predicate& pred, bool index_exists) const;
private:
    TableStatistics m_stats;
    static constexpr double CPU_TUPLE_COST = 0.01;
    static constexpr double RANDOM_FETCH_COST = 0.1;
};
```

**Verification Tasks:**

1.  **Unique Primary Key Selectivity:** Given $|R| = 1,000,000$ and
    distinct count $V(\text{id}) = 1,000,000$, estimate selectivity for
    $\text{id} = 42$. Verify that $S = 10^{-6}$ and that the optimiser
    selects an Index Scan.
2.  **Low-Cardinality Scan Selection:** Given $V(\text{gender}) = 2$ on
    $|R| = 500,000$, evaluate $\text{gender} = \text{'F'}$. Verify that
    $S \approx 0.5$ and that the optimiser falls back to a sequential
    table scan even if an index is present.
3.  **Numeric Range Estimation:** On an attribute with bounds
    $[0.0, 1000.0]$, estimate selectivity for $A > 800.0$. Verify that
    calculated selectivity is $0.20 \pm 0.01$.

#### Exercise 16.8: Composite Multi-Column Indexing

**Context and Theory:**

Real-world queries frequently filter across multiple attributes
simultaneously:

``` sql
SELECT * FROM Engineers WHERE dept = 'Systems' AND salary >= 9000.00;
```

Single-column indices force the engine to scan one index and evaluate
the remaining predicates sequentially in memory. A **Composite Index**
indexes a tuple of attributes $(A_1, A_2, \dots, A_m)$ using
**lexicographical ordering**:

$$(u_1, u_2) < (v_1, v_2) \iff (u_1 < v_1) \lor (u_1 = v_1 \land u_2 < v_2)$$

Composite indices obey the **Left-Prefix Rule**: an index over
$(A, B, C)$ accelerates queries filtering on $(A)$, $(A, B)$, and
$(A, B, C)$, but cannot accelerate queries filtering solely on $(B)$ or
$(C)$ without scanning the entire index (see fig. 12).

<figure id="fig:left-prefix">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1NzIuNjg4IiBo
ZWlnaHQ9IjU5Ljg3MiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA1NzIuNjg4IDU5Ljg3
MiI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxNHB4
Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFu
c3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgou
c3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewog
IGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxs
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5l
bmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVu
X2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Bl
bl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Jp
Z19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJv
dyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5
Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0i
MiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJl
dmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9s
eWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAg
OCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBv
cmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiBy
PSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBp
ZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtl
cldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4K
ICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2ly
Y2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdC
b3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWln
aHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIg
Y3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAg
PC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI1NzIu
Njg4IiBoZWlnaHQ9IjU5Ljg3MiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjE1NS4zOTQiIHk9IjM4LjQxMjAwMyIgPlByZWZpeDwvdGV4dD4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSIyMDQuMjgyMDEiIHk9IjM4LjQxMjAwMyIgPm1hdGNoPC90ZXh0Pgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMwOS4wNDIwMiIgeT0iMzguNDEyMDAzIiA+
UHJlZml4PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM1Ny45MzAwMiIg
eT0iMzguNDEyMDAzIiA+bWF0Y2g8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iNDYyLjY5IiB5PSIzOC40MTIwMDMiID5NaXNtYXRjaDwvdGV4dD4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSI4LjczMDAwMDUiIHk9IjEwLjQ3NiIgPkNvbXBvc2l0ZSBLZXkgVHVw
bGU6ICgmIzM5O1N5c3RlbXMmIzM5OywgODUwMC4wKSAmbHQ7ICgmIzM5O1N5c3RlbXMmIzM5Oywg
OTIwMC4wKSAmbHQ7ICgmIzM5O1dlYiYjMzk7LCA0MDAwLjApPC90ZXh0PgogIDxnPgogICAgPGxp
bmUgeDE9IjE2Ny42MTYiIHkxPSIxMy45NjgiIHgyPSIxNzQuNiIgeTI9IjI3LjkzNiIgY2xhc3M9
InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iMTc0LjYiIHkxPSIyNy45MzYiIHgyPSIyMTYu
NTA0MDEiIHkyPSIyNy45MzYiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjIy
My40ODgiIHkxPSIxMy45NjgiIHgyPSIyMTYuNTA0MDEiIHkyPSIyNy45MzYiIGNsYXNzPSJzb2xp
ZCI+PC9saW5lPgogIDwvZz4KICA8Zz4KICAgIDxsaW5lIHgxPSIzMjEuMjY0IiB5MT0iMTMuOTY4
IiB4Mj0iMzI4LjI0ODAyIiB5Mj0iMjcuOTM2IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxs
aW5lIHgxPSIzMjguMjQ4MDIiIHkxPSIyNy45MzYiIHgyPSIzNzAuMTUyIiB5Mj0iMjcuOTM2IiBj
bGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSIzNzcuMTM2MDIiIHkxPSIxMy45Njgi
IHgyPSIzNzAuMTUyIiB5Mj0iMjcuOTM2IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8L2c+CiAg
PGc+CiAgICA8bGluZSB4MT0iNDc0LjkxMjAyIiB5MT0iMTMuOTY4IiB4Mj0iNDgxLjg5NjAzIiB5
Mj0iMjcuOTM2IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSI0ODEuODk2MDMi
IHkxPSIyNy45MzYiIHgyPSI0OTUuODY0IiB5Mj0iMjcuOTM2IiBjbGFzcz0ic29saWQiPjwvbGlu
ZT4KICAgIDxsaW5lIHgxPSI1MDIuODQ4MDIiIHkxPSIxMy45NjgiIHgyPSI0OTUuODY0IiB5Mj0i
MjcuOTM2IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 12: Left-Prefix Matching</figcaption>
</figure>

**Specification:**

Implement a `CompositeKey` type containing `std::vector<Value>` with
lexicographical comparison operators. Construct a `CompositeBSTIndex`
class capable of mapping multi-attribute keys to `RowID` sets.

``` cpp
class CompositeKey {
public:
    CompositeKey(std::vector<Value> values) : m_values(std::move(values)) {}
    bool operator<(const CompositeKey& other) const {
        return std::lexicographical_compare(
            m_values.begin(), m_values.end(),
            other.m_values.begin(), other.m_values.end()
        );
    }
    bool operator==(const CompositeKey& other) const { return m_values == other.m_values; }
private:
    std::vector<Value> m_values;
};

class CompositeBSTIndex {
public:
    void insert(const CompositeKey& key, RowID id);
    std::vector<RowID> find_exact(const CompositeKey& key) const;
    std::vector<RowID> find_prefix_range(const CompositeKey& prefix_low, 
                                         const CompositeKey& prefix_high) const;
};
```

**Verification Tasks:**

1.  **Lexicographical Ordering Invariant:** Insert composite keys
    `("Beta", 2)`, `("Alpha", 10)`, and `("Alpha", 5)`. Verify that an
    in-order traversal yields `("Alpha", 5)`, `("Alpha", 10)`, then
    `("Beta", 2)`.
2.  **Point Lookup on Multi-Key:** Insert $5,000$ rows into
    `(dept, salary)`. Query for `("Systems", 8500.50)`. Verify that only
    matching row IDs are returned.
3.  **Left-Prefix Range Scan:** Query for all engineers in
    `dept = "Systems"` across any salary range by scanning with bounds
    `["Systems", -\infty]` to `["Systems", +\infty]`. Verify that all
    department employees are retrieved without scanning non-matching
    departments.

#### Exercise 16.9: Write-Ahead Logging (WAL) and Crash Recovery

**Context and Theory:**

To satisfy the **Durability ($D$)** invariant of ACID without writing
entire memory pages to disk synchronously on every transaction commit,
databases use **Write-Ahead Logging (WAL)** governed by two invariants:

1.  **The WAL Invariant:** Dirty data pages in memory must never be
    flushed to non-volatile storage until the corresponding log records
    detailing the mutation have been flushed to disk.
2.  **Commit Durability:** A transaction is not committed until its
    `COMMIT` log entry is guaranteed to be persistent on non-volatile
    media.

Every log record is assigned a strictly increasing **Log Sequence Number
(LSN)** (as shown in fig. 13). During recovery following a catastrophic
power loss, the engine executes an ARIES-style recovery pass:

- **REDO Pass:** Replays all logged modifications forward from the last
  checkpoint to restore the buffer state.
- **UNDO Pass:** Scans backward to reverse the modifications of active
  transactions that never committed before the crash.

<figure id="fig:wal-strategy">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0ODEuODk2MDMi
IGhlaWdodD0iNTkuODcyIiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDQ4MS44OTYwMyA1
OS44NzIiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJj
bGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0
cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsK
ICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2
Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFt
aWx5OiBJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTog
MTRweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDog
dHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsK
fQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVk
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5v
ZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cg
ewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJy
b3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRf
ZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRf
bWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2Jv
YiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7
Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgj
b3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1h
cmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmln
X29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5z
dmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJs
KCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0i
YXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9
IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+
CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJl
Zlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFy
dC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48
L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0i
MCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0i
NyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0i
NCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJr
ZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBt
YXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJz
ZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48
L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2
aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2Vy
SGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9
IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2Vy
PgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0i
NDgxLjg5NjAzIiBoZWlnaHQ9IjU5Ljg3MiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjEuNzQ2IiB5PSIxMC40NzYiID5bTFNOPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjM2LjY2NiIgeT0iMTAuNDc2IiA+MTo8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTcuNjE4IiB5PSIxMC40NzYiID5UWDE8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iODUuNTU0IiB5PSIxMC40NzYiID5TVEFSVF08L3Rl
eHQ+CiAgPGxpbmUgeDE9IjEzMi42OTYiIHkxPSI2Ljk4NCIgeDI9IjE0Ni42NjQiIHkyPSI2Ljk4
NCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIxNDYuNjY0LDMuNDky
IDE1My42NDgwMSw2Ljk4NCAxNDYuNjY0LDEwLjQ3NiIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29u
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE2Mi4zNzgiIHk9IjEwLjQ3NiIgPltM
U048L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTk3LjI5OCIgeT0iMTAu
NDc2IiA+Mjo8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjE4LjI1IiB5
PSIxMC40NzYiID5UWDE8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjQ2
LjE4NiIgeT0iMTAuNDc2IiA+SU5TRVJUPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjI5NS4wNzQiIHk9IjEwLjQ3NiIgPlJvdyMxXTwvdGV4dD4KICA8bGluZSB4MT0iMzQy
LjIxNiIgeTE9IjYuOTg0IiB4Mj0iMzU2LjE4NDAyIiB5Mj0iNi45ODQiIGNsYXNzPSJzb2xpZCI+
PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMzU2LjE4NDAyLDMuNDkyIDM2My4xNjgsNi45ODQg
MzU2LjE4NDAyLDEwLjQ3NiIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjM3MS44OTgiIHk9IjEwLjQ3NiIgPltMU048L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDA2LjgxODAyIiB5PSIxMC40NzYiID4zOjwvdGV4
dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0MjcuNzcwMDIiIHk9IjEwLjQ3NiIg
PkNSQVNIIV08L3RleHQ+CiAgPHBvbHlnb24gcG9pbnRzPSIzODQuMTIwMDMsMzEuNDI4MDAxIDM3
Ny4xMzYwMiwzNC45MjAwMDIgMzg0LjEyMDAzLDM4LjQxMjAwMyIgY2xhc3M9ImZpbGxlZCI+PC9w
b2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuNzQ2IiB5PSIzOC40MTIw
MDMiID5SZWNvdmVyeTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI2NC42
MDIwMDUiIHk9IjM4LjQxMjAwMyIgPlN0cmF0ZWd5OjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSIxMzQuNDQyIiB5PSIzOC40MTIwMDMiID5VbmRvPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE2OS4zNjIiIHk9IjM4LjQxMjAwMyIgPlRYMTwvdGV4
dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxOTcuMjk4IiB5PSIzOC40MTIwMDMi
ID5tdXRhdGlvbjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyNTMuMTcw
MDEiIHk9IjM4LjQxMjAwMyIgPihSb2xsYmFjayBSb3cjMSk8L3RleHQ+CiAgPGc+CiAgICA8bGlu
ZSB4MT0iNDIyLjUzMiIgeTE9IjEzLjk2OCIgeDI9IjQyMi41MzIiIHkyPSIzNC45MjAwMDIiIGNs
YXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjM4NC4xMjAwMyIgeTE9IjM0LjkyMDAw
MiIgeDI9IjQyMi41MzIiIHkyPSIzNC45MjAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDwv
Zz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 13: Write-Ahead Log Stream</figcaption>
</figure>

**Specification:**

Construct a `WALManager` class that writes binary mutation records to an
append-only stream. Implement serialization and deserialization routines
alongside a `recover()` function that inspects the log, commits
completed transactions, and rolls back uncommitted dirty rows.

``` cpp
enum class LogType : uint8_t { Begin, Insert, Commit, Rollback };

struct LogRecord {
    uint64_t lsn;
    uint32_t tx_id;
    LogType type;
    RowID target_row;
    Row payload;
};

class WALManager {
public:
    explicit WALManager(std::iostream& log_stream);
    uint64_t append_record(uint32_t tx_id, LogType type, RowID row_id, Row data);
    void flush();
    void recover(Table& target_table);
private:
    std::iostream& m_stream;
    uint64_t m_current_lsn{1};
};
```

**Verification Tasks:**

1.  **Redo Committed Transaction:** Write an `Insert` record followed by
    a `Commit` record to the WAL stream. Initialise an empty `Table` and
    execute `recover()`. Verify that the inserted row exists in the
    restored table.
2.  **Undo Uncommitted Transaction:** Write an `Insert` record for a
    transaction that ends abruptly without a matching `Commit` record
    (simulating power failure). Run `recover()`. Verify that the
    uncommitted row is pruned from active storage.
3.  **LSN Monotonicity:** Generate $1,000$ concurrent WAL writes. Verify
    that every assigned LSN is strictly monotonically increasing
    ($LSN_i < LSN_{i+1}$) and that deserialised binary frames pass
    payload verification.

#### Exercise 16.10: Multi-Version Concurrency Control (MVCC) Engine

**Context and Theory:**

Lock-based concurrency control (such as Two-Phase Locking, 2PL) forces
readers to block writers and writers to block readers, degrading
throughput. **Multi-Version Concurrency Control (MVCC)** guarantees that
**readers never block writers, and writers never block readers**.

Under MVCC, mutating a row does not overwrite its physical memory.
Instead, the storage layer maintains multiple versions of each tuple
decorated with visibility metadata (as in fig. 14):

- **$x_{\min}$:** The Transaction ID that created/inserted this version.
- **$x_{\max}$:** The Transaction ID that deleted or superseded this
  version (or $\infty$ / active sentinel if valid).

A transaction with ID $T_{\text{read}}$ can view a tuple version if and
only if:

$$x_{\min} \le T_{\text{read}} \quad \land \quad (x_{\max} > T_{\text{read}} \lor x_{\max} = \text{unassigned})$$

<figure id="fig:tuple-chain">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0ODguODgiIGhl
aWdodD0iNDUuOTA0IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDQ4OC44OCA0NS45MDQi
PgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5z
dmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13
aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJv
a2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0
ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBJ
b3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTRweDsK
fQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJhbnNw
YXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoKLnN2
Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsKICBm
aWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmlsbCB7
CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewogIG1h
cmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cgewog
IG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlhbW9u
ZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2Vk
X2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuZW5k
X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAu
c3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0KCi5z
dmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Blbl9j
aXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1z
dGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29wZW5f
Y2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2Ig
LnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNiaWdf
b3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJyb3ci
IHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1h
cmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdv
biBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8
bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIi
IG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZl
cnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3BvbHln
b24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAwIDgg
OCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3Jp
ZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0i
MiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9
Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJX
aWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAg
ICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNs
ZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3Qm94
PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0
PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5
PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogIDwv
ZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNDg4Ljg4
IiBoZWlnaHQ9IjQ1LjkwNCI+PC9yZWN0PgogIDxsaW5lIHgxPSIyNTguNDA4MDIiIHkxPSI2Ljk4
NCIgeDI9IjI3Mi4zNzYiIHkyPSI2Ljk4NCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHln
b24gcG9pbnRzPSIyNzIuMzc2LDMuNDkyIDI3OS4zNjAwMiw2Ljk4NCAyNzIuMzc2LDEwLjQ3NiIg
Y2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxsaW5lIHgxPSIyNTguNDA4MDIiIHkxPSIyMC45
NTIiIHgyPSIyNzIuMzc2IiB5Mj0iMjAuOTUyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9s
eWdvbiBwb2ludHM9IjI3Mi4zNzYsMTcuNDYwMDAxIDI3OS4zNjAwMiwyMC45NTIgMjcyLjM3Niwy
NC40NDQiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIxLjc0NiIgeT0iMTAuNDc2IiA+VmVyc2lvbiBBOiBbIHhtaW46IDEwMCwgeG1heDog
MTA1IF08L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjg4LjA5IiB5PSIx
MC40NzYiID5EYXRhOiAmIzM5O0FkYSBMb3ZlbGFjZSYjMzk7PC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjEuNzQ2IiB5PSIyNC40NDQiID5WZXJzaW9uIEI6IFsgeG1pbjog
MTA1LCB4bWF4OiBpbmYgXTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIy
ODguMDkiIHk9IjI0LjQ0NCIgPkRhdGE6ICYjMzk7Q291bnRlc3Mgb2YgTG92ZWxhY2UmIzM5Ozwv
dGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 14: Tuple Version Chain for RowID #1</figcaption>
</figure>

**Specification:**

Modify `Row` to contain MVCC version headers ($x_{\min}$, $x_{\max}$).
Implement an `MVCCTable` that performs snapshot-isolated scans,
non-destructive updates (marking $x_{\max}$ on the old record and
appending a new record with $x_{\min} = T$), and row-level conflict
checks.

``` cpp
struct MVCCRowHeader {
    uint64_t xmin{0};
    uint64_t xmax{std::numeric_limits<uint64_t>::max()};
};

class MVCCTable {
public:
    RowID insert(Row row, uint64_t tx_id);
    bool update(RowID id, Row new_row, uint64_t tx_id);
    bool remove(RowID id, uint64_t tx_id);
    std::vector<Row> snapshot_scan(uint64_t tx_id) const;
private:
    std::vector<Row> m_data;
    std::vector<MVCCRowHeader> m_headers;
};
```

**Verification Tasks:**

1.  **Snapshot Isolation Visibility:** Transaction $T_{100}$ inserts a
    row. Transaction $T_{105}$ modifies that row. Verify that an active
    reader transaction operating at snapshot $T_{102}$ reads the
    original value, while a reader at $T_{106}$ reads the modified
    value.
2.  **Non-Repeatable Read Prevention:** Open transaction $T_1$.
    Concurrently insert $50$ rows via transaction $T_2$ and commit.
    Verify that rescanning inside $T_1$ yields the identical row count
    as its opening scan.
3.  **Write-Write Conflict Detection:** Attempt to modify the same
    active tuple concurrently from two transactions $T_a$ and $T_b$.
    Verify that the engine detects an active uncommitted writer
    ($x_{\max} \neq \infty$) and aborts the second transaction.

#### Exercise 16.11: Paged Buffer Pool Manager with CLOCK Eviction

**Context and Theory:**

Databases manipulate datasets that exceed available physical RAM. Memory
management is orchestrated by the **Buffer Pool Manager**, which caches
fixed-size frames (typically $4\text{ KB}$ or $8\text{ KB}$) mapped from
secondary storage.

When a page is requested:

1.  If present in the pool, its **Pin Count** is incremented, and an
    in-memory pointer is returned.
2.  If absent, an existing unpinned frame ($\text{Pin Count} = 0$) must
    be evicted. If that victim frame is **dirty** (modified), it must be
    flushed to disk before replacement.

Eviction is governed by the **CLOCK (Second-Chance)** algorithm: a
circular iterator checks frame reference bits (see fig. 15). If the bit
is $1$, it is cleared to $0$; if the bit is $0$ and unpinned, the frame
is selected as the eviction victim.

<figure id="fig:clock-algorithm">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0NjcuOTI4IiBo
ZWlnaHQ9IjQ1LjkwNCIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA0NjcuOTI4IDQ1Ljkw
NCI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxNHB4
Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFu
c3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgou
c3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewog
IGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxs
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5l
bmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVu
X2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Bl
bl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Jp
Z19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJv
dyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5
Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0i
MiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJl
dmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9s
eWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAg
OCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBv
cmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiBy
PSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBp
ZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtl
cldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4K
ICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2ly
Y2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdC
b3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWln
aHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIg
Y3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAg
PC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI0Njcu
OTI4IiBoZWlnaHQ9IjQ1LjkwNCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjEuNzQ2IiB5PSIxMC40NzYiID5DTE9DSzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSI0My42NSIgeT0iMTAuNDc2IiA+SGFuZDwvdGV4dD4KICA8bGluZSB4MT0iNzYu
ODI0MDA1IiB5MT0iNi45ODQiIHgyPSI5MC43OTIiIHkyPSI2Ljk4NCIgY2xhc3M9InNvbGlkIj48
L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSI5MC43OTIsMy40OTIgOTcuNzc2LDYuOTg0IDkwLjc5
MiwxMC40NzYiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8bGluZSB4MT0iMjkzLjMyOCIg
eTE9IjYuOTg0IiB4Mj0iMzA3LjI5NjAyIiB5Mj0iNi45ODQiIGNsYXNzPSJzb2xpZCI+PC9saW5l
PgogIDxwb2x5Z29uIHBvaW50cz0iMzA3LjI5NjAyLDMuNDkyIDMxNC4yOCw2Ljk4NCAzMDcuMjk2
MDIsMTAuNDc2IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPGxpbmUgeDE9IjI5My4zMjgi
IHkxPSIyMC45NTIiIHgyPSIzMDcuMjk2MDIiIHkyPSIyMC45NTIiIGNsYXNzPSJzb2xpZCI+PC9s
aW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMzA3LjI5NjAyLDE3LjQ2MDAwMSAzMTQuMjgsMjAuOTUy
IDMwNy4yOTYwMiwyNC40NDQiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIzMjMuMDEiIHk9IjI0LjQ0NCIgPkV2aWN0PC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM2NC45MTQiIHk9IjI0LjQ0NCIgPlZpY3RpbTwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0MTMuODAyIiB5PSIyNC40NDQi
ID5Gb3VuZCE8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTA2LjUwNjAw
NCIgeT0iMTAuNDc2IiA+WyBGcmFtZSAwOiBSZWY9MSwgUGluPTAgXTwvdGV4dD4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMjMuMDEiIHk9IjEwLjQ3NiIgPkNsZWFyIFJlZj0wLCBB
ZHZhbmNlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEwNi41MDYwMDQi
IHk9IjI0LjQ0NCIgPlsgRnJhbWUgMTogUmVmPTAsIFBpbj0wIF08L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 15: CLOCK Algorithm</figcaption>
</figure>

**Specification:**

Implement a `BufferPoolManager` maintaining an array of in-memory page
slots. Implement `fetch_page`, `unpin_page`, and the CLOCK replacement
policy, handling dirty page identification.

``` cpp
class BufferPoolManager {
public:
    static constexpr size_t PAGE_SIZE = 4096;
    explicit BufferPoolManager(size_t pool_size);
    char* fetch_page(size_t page_id);
    void unpin_page(size_t page_id, bool is_dirty);
    void flush_all();
private:
    struct Frame {
        size_t page_id{0};
        uint32_t pin_count{0};
        bool is_dirty{false};
        bool reference_bit{false};
        std::array<char, PAGE_SIZE> data;
    };
    std::vector<Frame> m_frames;
    size_t m_clock_hand{0};
    size_t find_victim_frame();
};
```

**Verification Tasks:**

1.  **Cache Hit Verification:** Request page $5$ repeatedly across $10$
    iterations. Verify through an internal tracking counter that
    secondary storage read is invoked exactly once.
2.  **Pinned Page Immortality:** Pin every frame in the pool
    ($\text{Pin Count} > 0$). Request an un-cached page ID. Verify that
    the eviction algorithm refuses to evict active frames and throws a
    `std::runtime_error("Buffer pool exhausted")`.
3.  **Dirty Writeback Invariant:** Fetch page $2$, mutate its memory,
    and unpin with `is_dirty = true`. Force eviction by requesting new
    pages. Verify that the system executes a physical disk write before
    overwriting the slot.

#### Exercise 16.12: Foreign Key Constraint Enforcement and Referential Cascades

**Context and Theory:**

The relational model enforces **Referential Integrity** through Foreign
Key constraints. Let child relation $R_c$ declare foreign key $A_c$
referencing primary key $A_p$ of parent relation $R_p$:

$$\forall t_c \in R_c : t_c[A_c] \in \pi_{A_p}(R_p) \lor t_c[A_c] = \text{NULL}$$

Mutations must safeguard this invariant (see fig. 16):

1.  **Child Insertion/Update:** An inserted child row must be rejected
    if its foreign key value does not exist within the parent primary
    key index.
2.  **Parent Deletion/Update Actions:**
    - `RESTRICT`: The deletion of a parent record is rejected if
      referenced by any child rows.
    - `CASCADE`: Deleting a parent row triggers automatic deletion of
      all dependent child records across the foreign key link.

<figure id="fig:referential-cascades">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0NjAuOTQ0IiBo
ZWlnaHQ9Ijg3LjgwOCIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA0NjAuOTQ0IDg3Ljgw
OCI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxNHB4
Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFu
c3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgou
c3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewog
IGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxs
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5l
bmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVu
X2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Bl
bl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Jp
Z19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJv
dyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5
Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0i
MiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJl
dmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9s
eWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAg
OCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBv
cmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiBy
PSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBp
ZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtl
cldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4K
ICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2ly
Y2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdC
b3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWln
aHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIg
Y3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAg
PC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI0NjAu
OTQ0IiBoZWlnaHQ9Ijg3LjgwOCI+PC9yZWN0PgogIDxwb2x5Z29uIHBvaW50cz0iMjAyLjUzNjAx
LDMuNDkyIDE5NS41NTIsNi45ODQgMjAyLjUzNjAxLDEwLjQ3NiIgY2xhc3M9ImZpbGxlZCI+PC9w
b2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI2MC4xNTQwMiIgeT0iMjQu
NDQ0IiA+Rm9yZWlnbjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMTYu
MDI2IiB5PSIyNC40NDQiID5LZXk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iMzQzLjk2MiIgeT0iMjQuNDQ0IiA+UmVmZXJlbmNlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjEuNzQ2IiB5PSI2Ni4zNDgiID5bPC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjE1LjcxNDAwMSIgeT0iNjYuMzQ4IiA+QWN0aW9uOjwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3MS41ODYiIHk9IjY2LjM0OCIgPkNBU0NB
REU8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTI3LjQ1OCIgeT0iNjYu
MzQ4IiA+RGVsZXRlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE3Ni4z
NDYwMSIgeT0iNjYuMzQ4IiA+UGFyZW50PC90ZXh0PgogIDxsaW5lIHgxPSIyMjMuNDg4IiB5MT0i
NjIuODU2MDAzIiB4Mj0iMjQ0LjQ0IiB5Mj0iNjIuODU2MDAzIiBjbGFzcz0ic29saWQiPjwvbGlu
ZT4KICA8cG9seWdvbiBwb2ludHM9IjI0NC40NCw1OS4zNjQwMDIgMjUxLjQyNDAxLDYyLjg1NjAw
MyAyNDQuNDQsNjYuMzQ4IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iMjYwLjE1NDAyIiB5PSI2Ni4zNDgiID5BdXRvbWF0aWNhbGx5PC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM1Ny45MzAwMiIgeT0iNjYuMzQ4
IiA+UHJ1bmVzPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQwNi44MTgw
MiIgeT0iNjYuMzQ4IiA+Q2hpbGQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iNDQ4LjcyMjAyIiB5PSI2Ni4zNDgiID5dPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjEuNzQ2IiB5PSIxMC40NzYiID5QYXJlbnQ6IERlcGFydG1lbnQgKGlkOiAxKTwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxLjc0NiIgeT0iMzguNDEyMDAz
IiA+Q2hpbGQ6ICBFbmdpbmVlciAoZGVwdF9pZDogMSk8L3RleHQ+CiAgPGc+CiAgICA8bGluZSB4
MT0iMjAyLjUzNjAxIiB5MT0iNi45ODQiIHgyPSIyNDcuOTMyIiB5Mj0iNi45ODQiIGNsYXNzPSJz
b2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjI0Ny45MzIiIHkxPSI2Ljk4NCIgeDI9IjI0Ny45
MzIiIHkyPSIzNC45MjAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjIx
Ni41MDQwMSIgeTE9IjM0LjkyMDAwMiIgeDI9IjI0Ny45MzIiIHkyPSIzNC45MjAwMDIiIGNsYXNz
PSJzb2xpZCI+PC9saW5lPgogIDwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 16: Referential Integrity through
Cascades</figcaption>
</figure>

**Specification:**

Construct a `ForeignKeyConstraint` descriptor linking two `Table`
instances. Implement enforcement interceptors within `insert_record` and
`delete_records` supporting both `RESTRICT` and `CASCADE` policies.

``` cpp
enum class ReferentialAction { Restrict, Cascade, SetNull };

struct ForeignKeyConstraint {
    std::string child_col;
    Table& parent_table;
    std::string parent_pk_col;
    ReferentialAction on_delete;
};

class ConstraintManager {
public:
    void register_fk(Table& child_table, ForeignKeyConstraint fk);
    void validate_child_insert(const Table& child_table, const Row& row);
    void handle_parent_delete(Table& parent_table, const Row& deleted_row);
private:
    std::unordered_map<std::string, std::vector<ForeignKeyConstraint>> m_fk_registry;
};
```

**Verification Tasks:**

1.  **Invalid Foreign Key Rejection:** Attempt to insert an employee
    with `dept_id = 999` when the department table contains only IDs
    $\{1, 2, 3\}$. Verify that the manager rejects the insert with a
    `ConstraintViolationException`.
2.  **Restrict Policy Enforcement:** Attempt to delete department
    `id = 1` while active child engineer records reference it under a
    `RESTRICT` policy. Verify that the parent deletion is aborted and
    the table remains unmodified.
3.  **Cascading Child Pruning:** Configure a foreign key with `CASCADE`.
    Delete department `id = 1`. Verify that the parent record is deleted
    and that all child engineer records with `dept_id = 1` are expunged
    from the child table and its indices.